# 50. 다중 문제 충돌 조정

## 목적
한 분자에 toxicophore 문제가 여러 개 있을 때, 어떤 순서로 고칠지를
"리스트 순서대로"가 아니라 "먼저 고쳤을 때 남는 전체 문제 수가 가장
적어지는 순서"로 그리디하게 정하도록 iterative_fix_loop의 규칙 기반
분기를 개선한다.

## 배경
- 예선 제안서 제출 완료, 예선 통과 여부와 무관하게 지속 개선(오픈소스/대학원 포트폴리오) 진행 중
- 노트북 45: Aliphatic_long_chain 완전해결
- 노트북 46-47: 선례 라이브러리 11건 → 23건 확장 완료
- 노트북 48: LLM 호출 병렬화(batch_iterative_fix_loop) + 토의 기반 검증
  에이전트(ask_llm_debate_fix, proposer-critic 다라운드) 완료, iterative_fix_loop에
  use_debate 플래그로 연결
- 노트북 49: 도킹 자동화(docking.py, AutoDock Vina), 4개 표적(COMT/EGFR/NQO1×2)
  전부 방향성 재현 검증 완료. scoring.py(compute_multi_objective_score) 연동
  완료 — sascorer(외부저작물, .gitignore, 세션마다 다운로드) + Tox21 baseline
  모델(models/tox_baseline.py) 둘 다 debate critic 프롬프트에 자동 삽입되도록
  연결, 실제 동작 검증 완료
- Qwen(qwen3.8-max, DashScope token-plan) 주간 quota 소진, 8/15 15:37 UTC 리셋
  예정 — 그 전까지는 API 호출 없이 진행 가능한 작업 위주로 진행 중
- 남은 작업: 이번 세션(다중문제 충돌조정) → 감사추적 리포트 → test set
  1회 사용 시점 관리 → quota 리셋 후 전체 통합 파이프라인 실API 검증

## 이번 세션 목표
1. molecule_editor.py의 규칙 순서 결정 로직을 그리디 충돌조정 방식으로 교체
2. debate 왕복 기록(rounds)을 history에 저장하도록 수정
3. src/tools/audit.py — 사람이 읽기 좋은 감사추적 리포트 생성 함수 작성
4. 회귀 테스트 + 여러 문제가 동시에 있는 분자로 실제 동작 확인

## 작업 스타일 (선호)
- 수정 사항이 20줄 미만이고 새 항목 추가가 아니면 전체 코드 블록 대신 삽입 위치만 설명
- 재로드 + 검증 코드는 항상 하나의 셀로 통합해서 제공
- 파일 수정 후에는: 문법 검증 → 재로드 → 확인 → 커밋 (매번 잊지 말 것)

In [1]:
# 셀1 - install
!pip install rdkit -q
!pip install chembl_webresource_client -q
!pip install fuzzywuzzy python-Levenshtein -q
!pip install PyTDC --no-deps -q
!pip install PyYAML tqdm requests -q
!pip install openai -q
!apt-get install -y openbabel -qq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.4/37.4 MB 34.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.2/55.2 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.8/70.8 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 157.6/157.6 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 102.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.2/154.2 kB 13.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
Selecting previously unselected package libboost-iostreams1.74.0:amd64.
(Reading database ... 118332 files and directories currently installed.)
Preparing to unpack .../libboost-iostreams1.74.0_1.74.0-14ubuntu3_amd64.deb ...
Unpacking libboost-iostreams1.74.0:amd64 (1.74.0-14ubuntu3) ...
Selecting previously unselected package libinchi1.
Preparing to unpack .../libinchi1_1.03+dfsg-4_amd64.deb ...
Unpack

In [2]:
# 셀2 - github token + clone + cd + pwd
from google.colab import userdata

token = userdata.get('GITHUB_TOKEN')
!git clone https://{token}@github.com/Dec32th/laidd-2026.git
%cd /content/laidd-2026
!pwd

Cloning into 'laidd-2026'...
remote: Enumerating objects: 798, done.
remote: Counting objects: 100% (258/258), done.
remote: Compressing objects: 100% (159/159), done.
remote: Total 798 (delta 157), reused 193 (delta 98), pack-reused 540 (from 1)
Receiving objects: 100% (798/798), 7.18 MiB | 3.07 MiB/s, done.
Resolving deltas: 100% (474/474), done.
/content/laidd-2026
/content/laidd-2026


In [3]:
# 셀3 - git config
!git config --global user.email "hyekyeong.w@gmail.com"
!git config --global user.name "Dec32th"

In [4]:
# 셀4 - import + 데이터 로드 + vina 설치
import importlib, json, ast, time, os
from collections import Counter
from rdkit import Chem
from chembl_webresource_client.new_client import new_client
import requests

import src.tools.replacement_library
import src.tools.molecule_editor
import src.tools.atom_editor
import src.tools.toxicophore_detector
import src.tools.precedent_library
import src.tools.agent
import src.tools.docking
import models.tox_baseline

from src.tools.data_prep import load_tox21_clean
from src.tools.toxicophore_detector import detect_toxicophores
from src.tools.replacement_library import get_replacement_candidates
from src.tools.molecule_editor import propose_fix, iterative_fix_loop, clear_failure_memory
from src.tools.precedent_library import PRECEDENT_LIBRARY, get_precedents
from src.tools.docking import auto_dock_precedent, inspect_hetatm, DOCKING_TARGETS
from models.tox_baseline import train_tox21_baseline, make_tox_predictor

data = load_tox21_clean(random_state=7)
molecule = new_client.molecule

base_url = "https://www.guidetopharmacology.org/services"
def search_ligand(name):
    resp = requests.get(f"{base_url}/ligands", params={"name": name})
    return resp.json()
def get_ligand_interactions(ligand_id):
    resp = requests.get(f"{base_url}/ligands/{ligand_id}/interactions")
    return resp.json()

!wget -q https://github.com/ccsb-scripps/AutoDock-Vina/releases/download/v1.2.5/vina_1.2.5_linux_x86_64 -O vina_bin
!chmod +x vina_bin
print("vina_bin 존재:", os.path.exists('vina_bin'))

print(f"선례 수: {len(PRECEDENT_LIBRARY)} (23이어야 정상)")
print("agent.py 토의 로직 반영 여부:", 'ask_llm_debate_fix' in open('src/tools/agent.py').read())
print("agent.py 도킹 연결 반영 여부:", '_try_get_docking_evidence' in open('src/tools/agent.py').read())
print("agent.py scoring 연결 반영 여부:", '_try_compute_score' in open('src/tools/agent.py').read())
print("molecule_editor.py use_debate 반영 여부:", 'use_debate' in open('src/tools/molecule_editor.py').read())

[04:01:16] WARNING: not removing hydrogen atom without neighbors
[04:01:16] Explicit valence for atom # 8 Al, 6, is greater than permitted
[04:01:16] Explicit valence for atom # 3 Al, 6, is greater than permitted
[04:01:16] Explicit valence for atom # 4 Al, 6, is greater than permitted
[04:01:17] Explicit valence for atom # 4 Al, 6, is greater than permitted
[04:01:17] Explicit valence for atom # 9 Al, 6, is greater than permitted
[04:01:17] Explicit valence for atom # 5 Al, 6, is greater than permitted
[04:01:17] Explicit valence for atom # 16 Al, 6, is greater than permitted
[04:01:17] Explicit valence for atom # 20 Al, 6, is greater than permitted


전체: 7831개, 파싱 성공: 7823개, 파싱 실패(제외): 8개


[04:01:18] WARNING: not removing hydrogen atom without neighbors


vina_bin 존재: True
선례 수: 24 (23이어야 정상)
agent.py 토의 로직 반영 여부: True
agent.py 도킹 연결 반영 여부: True
agent.py scoring 연결 반영 여부: True
molecule_editor.py use_debate 반영 여부: True


In [5]:
# 셀5 - Qwen 연결 확인
from openai import OpenAI

dashscope_key = userdata.get('DASHSCOPE_API_KEY')
client_qwen = OpenAI(
    api_key=dashscope_key,
    base_url="https://token-plan.ap-southeast-1.maas.aliyuncs.com/compatible-mode/v1",
    timeout=30,
)
QWEN_MODEL = "qwen3.8-max"

try:
    response = client_qwen.chat.completions.create(
        model=QWEN_MODEL, messages=[{"role": "user", "content": "hi"}], max_tokens=10,
    )
    print("Qwen 연결 확인:", response.choices[0].message.content)
except Exception as e:
    print("Qwen 연결 실패 (quota 소진 가능성):", repr(e))

Qwen 연결 확인: Hi! How can I help you today?


In [14]:
!cat models/tox_baseline.py

"""Tox21 baseline 독성 예측 모델. 매 세션 직접 학습해서 사용(random_state
고정으로 재현성 확보). debate 로직의 tox_delta 콜백에 연결하기 위한 래퍼."""

import numpy as np
from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator
from sklearn.ensemble import RandomForestClassifier

_generator = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)


def smiles_to_ecfp(smiles):
    mol = Chem.MolFromSmiles(smiles)
    return _generator.GetFingerprintAsNumPy(mol) if mol else None


def train_tox21_baseline(data):
    """data: load_tox21_clean() 반환 딕셔너리.
    반환: {"classifiers": dict, "task_cols": list} — 이 딕셔너리 자체가
    학습된 모델 전체이며, 세션 내내 이 변수 하나만 들고 다니면 됨."""
    X_train, y_train, w_train = data['X_train'], data['y_train'], data['w_train']
    task_cols = data['task_cols']
    classifiers = {}
    for i, task in enumerate(task_cols):
        train_mask = w_train[:, i] == 1
        clf = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
        clf.fit(X_train[train_mask], y_

In [27]:
%%writefile src/tools/molecule_editor.py

from rdkit import Chem
from rdkit.Chem import rdMMPA
from src.tools.replacement_library import get_replacement_candidates
import hashlib
from src.tools.agent import (ask_llm_which_problem_to_fix, ask_llm_which_candidate_to_use,
                                  ask_llm_debate_fix, should_debate)

def _library_version_hash():
    from src.tools.replacement_library import get_replacement_candidates
    lib = get_replacement_candidates.__globals__['REPLACEMENT_LIBRARY']
    content_str = str(sorted(lib.items()))
    return hashlib.md5(content_str.encode()).hexdigest()[:8]

_FAILURE_MEMORY = {}


def clear_failure_memory():
    global _FAILURE_MEMORY
    _FAILURE_MEMORY = {}


def _check_and_match(part_smiles, problem_pattern, pattern_size):
    part_mol = Chem.MolFromSmiles(part_smiles.replace('[*:1]', 'C').replace('[*:2]', 'C'))
    if part_mol is None or not part_mol.HasSubstructMatch(problem_pattern):
        return False
    n_attachment = part_smiles.count('[*:')
    return part_mol.GetNumHeavyAtoms() - n_attachment == pattern_size


def find_core_and_target(smiles: str, rule_name: str):
    info = get_replacement_candidates(rule_name)
    if info is None:
        return None

    problem_pattern = Chem.MolFromSmarts(info['problem_smarts'])
    pattern_size = problem_pattern.GetNumAtoms()

    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None

    fragments1 = rdMMPA.FragmentMol(mol, maxCuts=1, resultsAsMols=False)
    for core, chain in fragments1:
        if core:
            continue
        parts = chain.split('.')
        if len(parts) != 2:
            continue
        for i, part in enumerate(parts):
            if _check_and_match(part, problem_pattern, pattern_size):
                return {"core": parts[1 - i], "target_removed": part}

    fragments2 = rdMMPA.FragmentMol(mol, maxCuts=2, resultsAsMols=False)
    for core, chain in fragments2:
        if not core:
            continue
        chain_parts = chain.split('.')
        if len(chain_parts) != 2:
            continue
        for i, part in enumerate(chain_parts):
            if not _check_and_match(part, problem_pattern, pattern_size):
                continue
            other_chain_part = chain_parts[1 - i]
            target_ap = '[*:1]' if '[*:1]' in part else ('[*:2]' if '[*:2]' in part else None)
            if target_ap is None:
                continue
            core_mol = Chem.MolFromSmiles(core)
            other_mol = Chem.MolFromSmiles(other_chain_part)
            if core_mol is None or other_mol is None:
                continue
            try:
                merged = Chem.molzip(core_mol, other_mol)
            except Exception:
                continue
            merged_smiles = Chem.MolToSmiles(merged)
            if merged_smiles.count('[*:') != 1:
                continue
            if '[*:1]' not in merged_smiles:
                merged_smiles = merged_smiles.replace('[*:2]', '[*:1]')
            return {"core": merged_smiles, "target_removed": part}

    return None


def reassemble_molecule(core_smiles: str, rule_name: str, candidate_idx: int = 0):
    info = get_replacement_candidates(rule_name)
    if info is None or candidate_idx >= len(info['candidates']):
        return None
    candidate = info['candidates'][candidate_idx]

    core_mol = Chem.MolFromSmiles(core_smiles)
    replacement_mol = Chem.MolFromSmiles(f"[*:1]{candidate['smiles']}")
    if core_mol is None or replacement_mol is None:
        return None

    try:
        combined = Chem.molzip(core_mol, replacement_mol)
        new_smiles = Chem.MolToSmiles(combined)
    except Exception:
        return None

    is_valid = Chem.MolFromSmiles(new_smiles) is not None

    return {
        "new_smiles": new_smiles,
        "candidate_used": candidate['name'],
        "rationale": candidate['rationale'],
        "is_valid": is_valid,
    }


def propose_fix(smiles: str, rule_name: str, candidate_idx: int = 0):
    info = get_replacement_candidates(rule_name)
    if info is None:
        return None

    if info.get("edit_method") == "atom_edit":
        from src.tools.atom_editor import apply_atom_edit_from_rule
        return apply_atom_edit_from_rule(smiles, rule_name, candidate_idx)

    located = find_core_and_target(smiles, rule_name)
    if located is None:
        return None
    return reassemble_molecule(located['core'], rule_name, candidate_idx)


def canonicalize(smiles: str):
    mol = Chem.MolFromSmiles(smiles)
    return Chem.MolToSmiles(mol) if mol else None


def _candidate_order_for_rule(rule_name: str, preferred_idx: int):
    info = get_replacement_candidates(rule_name)
    if info is None:
        return [preferred_idx]
    n = len(info['candidates'])
    order = [preferred_idx] if 0 <= preferred_idx < n else []
    order += [i for i in range(n) if i != preferred_idx]
    return order


def iterative_fix_loop(smiles: str, max_iterations: int = 10, candidate_idx: int = 0,
                        llm_client=None, llm_model=None, llm_client_type="gemini",
                        use_failure_memory: bool = True, use_debate: bool = False,
                        debate_max_rounds: int = 2):
    """진단->치환->재평가를 반복.
    핵심: candidate가 '화학적으로 유효(is_valid)'해도 대상 규칙이 실제로
    해소됐는지 재진단(detect_toxicophores)까지 확인한다. 그렇지 않으면
    항상 valid하지만 문제를 안 고치는 candidate(예: 단순 삽입형)가
    무한 반복 채택되어 진짜 해법(예: 분기형)으로 넘어가지 못하는 문제가
    있었음. 완전 해소가 안 되면 마지막으로 시도한(=대개 더 나은)
    valid 결과를 fallback으로 채택해 다음 iteration에서 계속 개선."""
    from src.tools.toxicophore_detector import detect_toxicophores
    from src.tools.agent import ask_llm_which_problem_to_fix, ask_llm_which_candidate_to_use

    current = canonicalize(smiles)
    seen = {current}
    history = [{"step": 0, "smiles": current}]
    skipped_rules = []
    skipped_details = []
    flagged_for_review = set()

    for step in range(1, max_iterations + 1):
        problems = detect_toxicophores(current)
        history[-1]["problems"] = problems

        if not problems:
            return {"status": "success", "final_smiles": current, "history": history,
                    "skipped_rules": skipped_rules, "skipped_details": skipped_details}

        known_problems = [p for p in problems if get_replacement_candidates(p['rule_name']) is not None
                          and p['rule_name'] not in flagged_for_review]
        unknown_problems = [p for p in problems if get_replacement_candidates(p['rule_name']) is None]

        for p in unknown_problems:
            if p['rule_name'] not in skipped_rules:
                skipped_rules.append(p['rule_name'])
                mol_cur = Chem.MolFromSmiles(current)
                matched_atoms = p['atom_indices']
                atom_symbols = [mol_cur.GetAtomWithIdx(i).GetSymbol() for i in matched_atoms] if mol_cur else []
                skipped_details.append({
                    "rule_name": p['rule_name'],
                    "reason": f"라이브러리에 등록되지 않은 규칙입니다. FilterCatalog(PAINS/BRENK)가 "
                              f"'{p['rule_name']}'로 진단했으며, 매치된 원자 인덱스는 {matched_atoms}"
                              f"(원소: {atom_symbols})입니다. 이 구조에 대한 치환 규칙을 "
                              f"replacement_library.py에 추가하면 자동으로 처리 가능합니다.",
                    "atom_indices": matched_atoms,
                })

        if not known_problems:
            return {"status": "no_known_fix", "final_smiles": current, "history": history,
                    "skipped_rules": skipped_rules, "skipped_details": skipped_details}

        if llm_client is not None:
            problem_decision = ask_llm_which_problem_to_fix(llm_client, llm_model, current, problems, client_type=llm_client_type)
            preferred_rule = problem_decision['rule_name']
            problem_reason = problem_decision.get('reason', '')
            ordered_rules = [preferred_rule] + [p['rule_name'] for p in known_problems if p['rule_name'] != preferred_rule]
        else:
            # 다중 문제 충돌 조정: 각 문제를 먼저 고쳤을 때 남는 전체
            # toxicophore 수가 가장 적어지는 순서로 정렬(그리디, LLM 미사용).
            sim_scores = {}
            for p in known_problems:
                rn = p['rule_name']
                try:
                    trial = propose_fix(current, rn, candidate_idx)
                    if trial is None or not trial.get('is_valid'):
                        sim_scores[rn] = 999
                        continue
                    remaining = detect_toxicophores(trial['new_smiles'])
                    sim_scores[rn] = len(remaining)
                except Exception:
                    sim_scores[rn] = 999
            ordered_rules = sorted(sim_scores, key=sim_scores.get)
            problem_reason = f"규칙 기반(충돌 조정: 남는 문제 수 적은 순 - {sim_scores})"

        fixed = None
        target_rule = None
        candidate_reason = None
        failed_attempts = []

        for candidate_rule in ordered_rules:
            debate_log_for_step = None
            if llm_client is not None:
                candidate_decision = ask_llm_which_candidate_to_use(llm_client, llm_model, current, candidate_rule, client_type=llm_client_type)
                preferred_candidate_idx = candidate_decision['candidate_idx']
                this_candidate_reason = candidate_decision.get('reason', '')

                if preferred_candidate_idx == -1:
                    flagged_for_review.add(candidate_rule)
                    if candidate_rule not in skipped_rules:
                        skipped_rules.append(candidate_rule)
                    skipped_details.append({
                        "rule_name": candidate_rule,
                        "reason": f"LLM이 치환을 보류했습니다: {this_candidate_reason} "
                                  f"(이 분자가 [참고] 사항에 해당하는 안전한 실사용 사례와 유사하다고 "
                                  f"판단되어, 자동 치환 대신 연구자의 직접 검토를 권장합니다.)",
                        "atom_indices": next((p['atom_indices'] for p in problems if p['rule_name'] == candidate_rule), []),
                    })
                    continue
            else:
                preferred_candidate_idx = candidate_idx
                this_candidate_reason = "규칙 기반(고정 인덱스 우선, 실패/미해소 시 같은 규칙 내 다른 candidate로 재시도)"

            rule_fixed = None
            fallback_attempt = None
            fallback_used_idx = None
            fallback_reason = None

            for try_idx in _candidate_order_for_rule(candidate_rule, preferred_candidate_idx):
                memory_key = (current, candidate_rule, try_idx, _library_version_hash())
                if use_failure_memory and memory_key in _FAILURE_MEMORY:
                    failed_attempts.append(f"{candidate_rule}[idx={try_idx}](memory-skip)")
                    continue

                attempt = propose_fix(current, candidate_rule, try_idx)
                if attempt is None or not attempt.get('is_valid'):
                    failed_attempts.append(f"{candidate_rule}[idx={try_idx}]")
                    if use_failure_memory:
                        _FAILURE_MEMORY[memory_key] = True
                    continue

                # valid해도 실제로 이 규칙이 재진단에서 사라졌는지 확인
                recheck = detect_toxicophores(attempt['new_smiles'])
                still_flagged = any(p['rule_name'] == candidate_rule for p in recheck)

                if not still_flagged:
                    candidate_obj = get_replacement_candidates(candidate_rule)['candidates'][try_idx]
                    debate_suffix = ""

                    if use_debate and llm_client is not None and should_debate(candidate_obj.get('rationale', '')):
                        debate_result = ask_llm_debate_fix(
                            llm_client, llm_model, current, attempt['new_smiles'], candidate_rule,
                            candidate_obj['name'], candidate_obj.get('rationale', ''),
                            client_type=llm_client_type, max_rounds=debate_max_rounds,
                        )
                        if debate_result['final_verdict'] == 'rejected':
                            failed_attempts.append(f"{candidate_rule}[idx={try_idx}](토의 결과 반려)")
                            if use_failure_memory:
                                _FAILURE_MEMORY[memory_key] = True
                            continue
                        elif debate_result['final_verdict'] == 'escalate':
                            flagged_for_review.add(candidate_rule)
                            if candidate_rule not in skipped_rules:
                                skipped_rules.append(candidate_rule)
                            skipped_details.append({
                                "rule_name": candidate_rule,
                                "reason": f"LLM 토의가 {debate_max_rounds}라운드 안에 합의에 도달하지 못해 "
                                          f"사람 검토로 넘김 (마지막 논쟁: {debate_result['rounds'][-1]['text']})",
                                "atom_indices": next((p['atom_indices'] for p in problems if p['rule_name'] == candidate_rule), []),
                            })
                            failed_attempts.append(f"{candidate_rule}[idx={try_idx}](토의 합의 실패, escalate)")
                            continue
                        debate_log_for_step = debate_result['rounds']
                        debate_suffix = " (토의 승인)"

                    rule_fixed = attempt
                    candidate_reason = f"{this_candidate_reason} (candidate_idx={try_idx}, 완전 해소){debate_suffix}"
                    break
                else:
                    failed_attempts.append(f"{candidate_rule}[idx={try_idx}](valid이나 미해소)")
                    fallback_attempt = attempt
                    fallback_used_idx = try_idx
                    fallback_reason = f"{this_candidate_reason} (candidate_idx={try_idx}, 부분 개선/다음 iteration에서 계속)"

            if rule_fixed is None and fallback_attempt is not None:
                rule_fixed = fallback_attempt
                candidate_reason = fallback_reason

            if rule_fixed is not None:
                fixed = rule_fixed
                target_rule = candidate_rule
                break

        if fixed is None:
            reason_detail = (f"이 단계에서 known 규칙들의 모든 candidate를 순서대로 시도했으나 "
                              f"({failed_attempts}) 모두 실행에 실패했습니다(memory-skip 표시는 이전에 "
                              f"실패했던 것으로 확인되어 재시도 없이 건너뛴 항목). 흔한 원인: 유기금속/무기염 "
                              f"등 특수 화학종, 고리 구조와의 예상치 못한 충돌, 또는 원자가 계산 오류입니다.")
            return {"status": "stuck", "reason": f"시도한 규칙/candidate {failed_attempts} 모두 치환 실패",
                    "reason_detail": reason_detail,
                    "final_smiles": current, "history": history,
                    "skipped_rules": skipped_rules, "skipped_details": skipped_details}

        new_current = canonicalize(fixed['new_smiles'])

        if new_current in seen:
            return {"status": "cycle_detected", "final_smiles": current, "history": history,
                    "skipped_rules": skipped_rules, "skipped_details": skipped_details}

        seen.add(new_current)
        current = new_current
        history.append({
            "step": step,
            "smiles": current,
            "fixed_rule": target_rule,
            "problem_reason": problem_reason,
            "candidate_used": fixed['candidate_used'],
            "candidate_reason": candidate_reason,
            "debate_rounds": debate_log_for_step,
        })

    return {"status": "max_iterations_reached", "final_smiles": current, "history": history,
            "skipped_rules": skipped_rules, "skipped_details": skipped_details}


def batch_iterative_fix_loop(smiles_list, max_iterations=10, candidate_idx=0,
                               llm_client=None, llm_model=None, llm_client_type="gemini",
                               max_workers=5, progress=True, use_debate=False,
                               debate_max_rounds=2):
    """여러 분자에 iterative_fix_loop를 스레드 병렬로 적용.
    LLM API 호출이 병목인 경우(네트워크 대기 시간) 유효한 개선이며,
    화학 계산 로직(iterative_fix_loop 자체)은 전혀 수정하지 않는다.
    반환: [(smiles, result_dict), ...] (완료 순서, 입력 순서와 다를 수 있음)
    """
    from concurrent.futures import ThreadPoolExecutor, as_completed

    def _process_one(smi):
        r = iterative_fix_loop(
            smi, max_iterations=max_iterations, candidate_idx=candidate_idx,
            llm_client=llm_client, llm_model=llm_model, llm_client_type=llm_client_type,
            use_debate=use_debate, debate_max_rounds=debate_max_rounds,
        )
        return smi, r

    results = []
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(_process_one, smi): smi for smi in smiles_list}
        for i, future in enumerate(as_completed(futures)):
            smi, r = future.result()
            results.append((smi, r))
            if progress:
                print(f"[{i+1}/{len(smiles_list)}] {smi[:30]} -> {r['status']}")
    return results


def batch_iterative_fix_loop(smiles_list, max_iterations=10, candidate_idx=0,
                               llm_client=None, llm_model=None, llm_client_type="gemini",
                               max_workers=5, progress=True):
    """여러 분자에 iterative_fix_loop를 스레드 병렬로 적용.
    LLM API 호출이 병목인 경우(네트워크 대기 시간) 유효한 개선이며,
    화학 계산 로직(iterative_fix_loop 자체)은 전혀 수정하지 않는다.
    반환: [(smiles, result_dict), ...] (완료 순서, 입력 순서와 다를 수 있음)
    """
    from concurrent.futures import ThreadPoolExecutor, as_completed

    def _process_one(smi):
        r = iterative_fix_loop(
            smi, max_iterations=max_iterations, candidate_idx=candidate_idx,
            llm_client=llm_client, llm_model=llm_model, llm_client_type=llm_client_type,
        )
        return smi, r

    results = []
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(_process_one, smi): smi for smi in smiles_list}
        for i, future in enumerate(as_completed(futures)):
            smi, r = future.result()
            results.append((smi, r))
            if progress:
                print(f"[{i+1}/{len(smiles_list)}] {smi[:30]} -> {r['status']}")
    return results


Overwriting src/tools/molecule_editor.py


In [8]:
%%writefile src/tools/audit.py
"""iterative_fix_loop 결과를 사람이 읽기 좋은 감사추적 리포트로 변환.
심사/발표 자료용 — AI가 왜 그렇게 판단했는지, 언제 사람 검토로 넘겼는지를
그대로 보여준다."""


def generate_audit_report(result, original_smiles=None):
    lines = []
    lines.append("=" * 60)
    lines.append("치환 감사추적 리포트")
    lines.append("=" * 60)
    if original_smiles:
        lines.append(f"원본 분자: {original_smiles}")
    lines.append(f"최종 상태: {result['status']}")
    lines.append(f"최종 분자: {result.get('final_smiles', '')}")
    lines.append("")

    lines.append("--- 단계별 이력 ---")
    for h in result.get('history', []):
        step = h.get('step')
        lines.append(f"\n[스텝 {step}] {h.get('smiles', '')}")
        problems = h.get('problems')
        if problems:
            rule_names = [p['rule_name'] for p in problems]
            lines.append(f"  진단된 문제: {rule_names}")
        if 'fixed_rule' in h:
            lines.append(f"  고친 규칙: {h['fixed_rule']} (판단 근거: {h.get('problem_reason', '')})")
            lines.append(f"  적용된 치환: {h.get('candidate_used', '')}")
            lines.append(f"  candidate 선택 근거: {h.get('candidate_reason', '')}")
            debate_rounds = h.get('debate_rounds')
            if debate_rounds:
                lines.append("  --- 토의(debate) 왕복 기록 ---")
                for r in debate_rounds:
                    role = r.get('role')
                    round_num = r.get('round')
                    text = r.get('text', {})
                    if role == 'critic':
                        lines.append(f"    [R{round_num} critic] {text.get('verdict')}: {text.get('reason', '')}")
                    else:
                        lines.append(f"    [R{round_num} proposer] {text.get('stance')}: {text.get('argument', '')}")

    if result.get('skipped_rules'):
        lines.append("\n--- 처리 못 하고 넘긴 규칙 (라이브러리 미등록 또는 사람 검토 필요) ---")
        for detail in result.get('skipped_details', []):
            lines.append(f"  - {detail['rule_name']}: {detail['reason']}")

    if result['status'] == 'stuck':
        lines.append(f"\n--- stuck 사유 ---\n{result.get('reason_detail', result.get('reason', ''))}")

    lines.append("=" * 60)
    return "\n".join(lines)

Writing src/tools/audit.py


In [9]:
import ast
with open('src/tools/molecule_editor.py') as f:
    content = f.read()
    ast.parse(content)
print("✅ molecule_editor.py 문법 정상")
print("✅ 충돌 조정 반영:", '충돌 조정' in content)
print("✅ debate_rounds 저장 반영:", 'debate_rounds' in content)

with open('src/tools/audit.py') as f:
    ast.parse(f.read())
print("✅ audit.py 문법 정상")

importlib.reload(src.tools.replacement_library)
importlib.reload(src.tools.atom_editor)
importlib.reload(src.tools.molecule_editor)
import src.tools.audit
importlib.reload(src.tools.audit)
from src.tools.molecule_editor import iterative_fix_loop, clear_failure_memory
from src.tools.audit import generate_audit_report
clear_failure_memory()

r = iterative_fix_loop('CCCCCCCCCCCCCCCC', max_iterations=10, candidate_idx=0)
assert r['status'] == 'success'
print("✅ 회귀 없음 확인")

print("\n" + generate_audit_report(r, original_smiles='CCCCCCCCCCCCCCCC'))

✅ molecule_editor.py 문법 정상
✅ 충돌 조정 반영: True
✅ debate_rounds 저장 반영: True
✅ audit.py 문법 정상
✅ 회귀 없음 확인

치환 감사추적 리포트
원본 분자: CCCCCCCCCCCCCCCC
최종 상태: success
최종 분자: CCCCC(C)CCCC(C)CCCC(C)CCC

--- 단계별 이력 ---

[스텝 0] CCCCCCCCCCCCCCCC
  진단된 문제: ['Aliphatic_long_chain']

[스텝 1] CCCCC(C)CCCC(C)CCCC(C)CCC
  고친 규칙: Aliphatic_long_chain (판단 근거: 규칙 기반(충돌 조정: 남는 문제 수 적은 순 - {'Aliphatic_long_chain': 1}))
  적용된 치환: multi-ether chain (multiple O inserted for long chains)
  candidate 선택 근거: 규칙 기반(고정 인덱스 우선, 실패/미해소 시 같은 규칙 내 다른 candidate로 재시도) (candidate_idx=1, 완전 해소)


In [11]:
for smi in data['smiles_valid'][:500]:
    problems = detect_toxicophores(smi)
    known = [p for p in problems if get_replacement_candidates(p['rule_name']) is not None]
    if len(known) >= 2:
        print(smi, [p['rule_name'] for p in known])
        break
multi_smi = "NNC(=O)CP(=O)(c1ccccc1)c1ccccc1"
clear_failure_memory()
r2 = iterative_fix_loop(multi_smi, max_iterations=10, candidate_idx=0)
print(generate_audit_report(r2, original_smiles=multi_smi))

NNC(=O)CP(=O)(c1ccccc1)c1ccccc1 ['hydrazine', 'phosphor']
치환 감사추적 리포트
원본 분자: NNC(=O)CP(=O)(c1ccccc1)c1ccccc1
최종 상태: stuck
최종 분자: NC(=O)CP(=O)(c1ccccc1)c1ccccc1

--- 단계별 이력 ---

[스텝 0] NNC(=O)CP(=O)(c1ccccc1)c1ccccc1
  진단된 문제: ['acyl_hydrazine', 'hydrazine', 'Oxygen-nitrogen_single_bond', 'phosphor']

[스텝 1] NC(=O)CP(=O)(c1ccccc1)c1ccccc1
  진단된 문제: ['phosphor']
  고친 규칙: hydrazine (판단 근거: 규칙 기반(충돌 조정: 남는 문제 수 적은 순 - {'hydrazine': 1, 'phosphor': 999}))
  적용된 치환: amide/amine (terminal N removed)
  candidate 선택 근거: 규칙 기반(고정 인덱스 우선, 실패/미해소 시 같은 규칙 내 다른 candidate로 재시도) (candidate_idx=0, 완전 해소)

--- 처리 못 하고 넘긴 규칙 (라이브러리 미등록 또는 사람 검토 필요) ---
  - acyl_hydrazine: 라이브러리에 등록되지 않은 규칙입니다. FilterCatalog(PAINS/BRENK)가 'acyl_hydrazine'로 진단했으며, 매치된 원자 인덱스는 [0, 1, 2, 3](원소: ['N', 'N', 'C', 'O'])입니다. 이 구조에 대한 치환 규칙을 replacement_library.py에 추가하면 자동으로 처리 가능합니다.
  - Oxygen-nitrogen_single_bond: 라이브러리에 등록되지 않은 규칙입니다. FilterCatalog(PAINS/BRENK)가 'Oxygen-nitrogen_single_bond'로 진단했으며, 매치된 원자 인덱스는 [0, 1](원소: ['N',

In [12]:
!cd /content/laidd-2026 && git add -A && git commit -m "Add greedy multi-problem conflict resolution (fewest-remaining-problems ordering) and audit trail report generator" && git push

[main a146f45] Add greedy multi-problem conflict resolution (fewest-remaining-problems ordering) and audit trail report generator
 2 files changed, 69 insertions(+), 2 deletions(-)
 create mode 100644 src/tools/audit.py
Enumerating objects: 10, done.
Counting objects: 100% (10/10), done.
Delta compression using up to 2 threads
Compressing objects: 100% (6/6), done.
Writing objects: 100% (6/6), 1.99 KiB | 1.99 MiB/s, done.
Total 6 (delta 3), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (3/3), completed with 3 local objects.
To https://github.com/Dec32th/laidd-2026.git
   591076d..a146f45  main -> main


In [13]:
%%writefile src/tools/activity_metrics.py
"""치환 전후 활성 보존 가능성을 정성/정량으로 평가하는 지표 모음.
2D 연결성(Tanimoto), 3D 형태(회전반경), QED/LogP/합성용이성 변화를
종합해 판정한다. sascorer는 외부 저작물이라 파라미터로 주입받는다
(scoring.py와 동일한 패턴)."""

from rdkit import Chem
from rdkit.Chem import Descriptors, Descriptors3D, DataStructs, QED, AllChem
from models.tox_baseline import smiles_to_fp_bitvect


def compute_activity_preservation_metrics(original_smiles, fixed_smiles, sascorer_module):
    mol_o = Chem.MolFromSmiles(original_smiles)
    mol_f = Chem.MolFromSmiles(fixed_smiles)
    if mol_o is None or mol_f is None:
        return None

    fp_o = smiles_to_fp_bitvect(original_smiles)
    fp_f = smiles_to_fp_bitvect(fixed_smiles)
    tanimoto = DataStructs.TanimotoSimilarity(fp_o, fp_f)

    qed_o, qed_f = QED.qed(mol_o), QED.qed(mol_f)
    logp_o, logp_f = Descriptors.MolLogP(mol_o), Descriptors.MolLogP(mol_f)
    sa_o = sascorer_module.calculateScore(mol_o)
    sa_f = sascorer_module.calculateScore(mol_f)

    def get_3d(mol):
        m = Chem.AddHs(mol)
        if AllChem.EmbedMolecule(m, randomSeed=42) != 0:
            return None
        AllChem.MMFFOptimizeMolecule(m)
        return m

    m3d_o, m3d_f = get_3d(mol_o), get_3d(mol_f)
    shape_available = m3d_o is not None and m3d_f is not None

    result = {"tanimoto": tanimoto, "delta_qed": qed_f - qed_o, "delta_logp": logp_f - logp_o,
              "delta_sa_score": sa_f - sa_o, "shape_available": shape_available}
    if shape_available:
        rog_o = Descriptors3D.RadiusOfGyration(m3d_o)
        rog_f = Descriptors3D.RadiusOfGyration(m3d_f)
        result["delta_rog_pct"] = (rog_f - rog_o) / rog_o * 100 if rog_o != 0 else None
    return result


def classify_activity_risk_v3(metrics):
    if metrics is None:
        return {"verdict": "판정 불가", "details": []}
    details, warnings = [], []

    conn_ok = metrics["tanimoto"] >= 0.5
    details.append(f"2D 연결성: {'유사' if conn_ok else '상이'} (Tanimoto {metrics['tanimoto']:.3f})")

    shape_ok = None
    if metrics["shape_available"] and metrics.get("delta_rog_pct") is not None:
        shape_ok = abs(metrics["delta_rog_pct"]) < 15
        details.append(f"3D 형태: {'보존' if shape_ok else '변화'} (회전반경 {metrics['delta_rog_pct']:+.1f}%)")
    else:
        details.append("3D 형태: 계산 불가")

    qed_ok = abs(metrics["delta_qed"]) < 0.1
    details.append(f"약물유사성(QED): {'유지' if qed_ok else '변화'} ({metrics['delta_qed']:+.3f})")
    if not qed_ok:
        warnings.append("QED 변화")

    logp_ok = abs(metrics["delta_logp"]) < 1.0
    details.append(f"소수성(LogP): {'유지' if logp_ok else '변화'} ({metrics['delta_logp']:+.3f})")
    if not logp_ok:
        warnings.append("LogP 변화")

    sa_ok = metrics["delta_sa_score"] < 0.5
    details.append(f"합성용이성(SA): {'유지/개선' if sa_ok else '악화'} ({metrics['delta_sa_score']:+.3f})")
    if not sa_ok:
        warnings.append("합성난이도 증가")

    if shape_ok is None:
        verdict = "3D 형태 계산 불가 — 2D 지표만으로 판단, 신뢰도 낮음"
    elif shape_ok:
        verdict = ("구조·형태 모두 보존 — 활성 유지 가능성 높음" if conn_ok else
                   "2D 연결성은 크게 바뀌었으나 3D 형태는 보존됨 (bioisostere 가능성) — 활성 유지 기대")
    else:
        verdict = "3D 형태 자체가 크게 변화 — 표적 결합 형태 훼손 우려, 사람 검토 필요"

    if warnings:
        verdict += f" [보조 경고: {', '.join(warnings)}]"

    return {"verdict": verdict, "details": details, "shape_ok": shape_ok, "warnings": warnings}

Writing src/tools/activity_metrics.py


In [15]:
%%writefile models/tox_baseline.py

"""Tox21 baseline 독성 예측 모델. 매 세션 직접 학습해서 사용(random_state
고정으로 재현성 확보). debate 로직의 tox_delta 콜백에 연결하기 위한 래퍼."""

import numpy as np
from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator
from sklearn.ensemble import RandomForestClassifier

_generator = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)


def smiles_to_ecfp(smiles):
    mol = Chem.MolFromSmiles(smiles)
    return _generator.GetFingerprintAsNumPy(mol) if mol else None

def smiles_to_fp_bitvect(smiles):
    mol = Chem.MolFromSmiles(smiles)
    return _generator.GetFingerprint(mol) if mol else None


def train_tox21_baseline(data):
    """data: load_tox21_clean() 반환 딕셔너리.
    반환: {"classifiers": dict, "task_cols": list} — 이 딕셔너리 자체가
    학습된 모델 전체이며, 세션 내내 이 변수 하나만 들고 다니면 됨."""
    X_train, y_train, w_train = data['X_train'], data['y_train'], data['w_train']
    task_cols = data['task_cols']
    classifiers = {}
    for i, task in enumerate(task_cols):
        train_mask = w_train[:, i] == 1
        clf = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
        clf.fit(X_train[train_mask], y_train[train_mask, i])
        classifiers[task] = clf
    return {"classifiers": classifiers, "task_cols": task_cols}


def predict_tox21_avg(smiles, model):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    fp = smiles_to_ecfp(smiles).reshape(1, -1)
    classifiers, task_cols = model["classifiers"], model["task_cols"]
    return float(np.mean([classifiers[t].predict_proba(fp)[0][1] for t in task_cols]))


def make_tox_predictor(model):
    """set_tox_predictor()에 바로 넘길 수 있는 콜백 생성.
    반환값은 (fixed 예측 - original 예측): 음수면 독성이 줄어든 것."""
    def _predict(original_smiles, fixed_smiles, rule_name):
        p_o = predict_tox21_avg(original_smiles, model)
        p_f = predict_tox21_avg(fixed_smiles, model)
        if p_o is None or p_f is None:
            return None
        return p_f - p_o
    return _predict


Overwriting models/tox_baseline.py


In [6]:
import urllib.request, sys
urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/rdkit/rdkit/master/Contrib/SA_Score/sascorer.py", "sascorer.py")
urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/rdkit/rdkit/master/Contrib/SA_Score/fpscores.pkl.gz", "fpscores.pkl.gz")
sys.path.append('.')
import sascorer

from src.tools.agent import set_sascorer_module
set_sascorer_module(sascorer)
print("sascorer 등록 완료")

sascorer 등록 완료


In [18]:
import ast
with open('src/tools/activity_metrics.py') as f:
    ast.parse(f.read())
print("✅ activity_metrics.py 문법 정상")

with open('models/tox_baseline.py') as f:
    content = f.read()
    ast.parse(content)
print("✅ smiles_to_fp_bitvect 반영:", 'smiles_to_fp_bitvect' in content)

import models.tox_baseline
importlib.reload(models.tox_baseline)
import src.tools.activity_metrics
importlib.reload(src.tools.activity_metrics)
from src.tools.activity_metrics import compute_activity_preservation_metrics, classify_activity_risk_v3
print("✅ import 성공")

metrics = compute_activity_preservation_metrics(
    "NCCc1ccc(O)c(O)c1", "COc1ccc(CCN)cc1O", sascorer
)
print(metrics)
print(classify_activity_risk_v3(metrics))

✅ activity_metrics.py 문법 정상
✅ smiles_to_fp_bitvect 반영: True
✅ import 성공
{'tanimoto': 0.6, 'delta_qed': 0.1580250816015243, 'delta_logp': 0.30300000000000016, 'delta_sa_score': -0.14674478887540765, 'shape_available': True, 'delta_rog_pct': 6.761129611827899}
{'verdict': '구조·형태 모두 보존 — 활성 유지 가능성 높음 [보조 경고: QED 변화]', 'details': ['2D 연결성: 유사 (Tanimoto 0.600)', '3D 형태: 보존 (회전반경 +6.8%)', '약물유사성(QED): 변화 (+0.158)', '소수성(LogP): 유지 (+0.303)', '합성용이성(SA): 유지/개선 (-0.147)'], 'shape_ok': True, 'warnings': ['QED 변화']}


In [19]:
!cd /content/laidd-2026 && git add -A && git commit -m "Consolidate activity preservation metrics (compute_activity_preservation_metrics, classify_activity_risk_v3) into src/tools/activity_metrics.py, reusing fingerprint from models/tox_baseline.py" && git push

[main 4c20dce] Consolidate activity preservation metrics (compute_activity_preservation_metrics, classify_activity_risk_v3) into src/tools/activity_metrics.py, reusing fingerprint from models/tox_baseline.py
 2 files changed, 91 insertions(+)
 create mode 100644 src/tools/activity_metrics.py
Enumerating objects: 12, done.
Counting objects: 100% (12/12), done.
Delta compression using up to 2 threads
Compressing objects: 100% (7/7), done.
Writing objects: 100% (7/7), 2.21 KiB | 2.21 MiB/s, done.
Total 7 (delta 4), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (4/4), completed with 4 local objects.
To https://github.com/Dec32th/laidd-2026.git
   a146f45..4c20dce  main -> main


In [7]:
!cat src/tools/agent.py

import json
import time
from src.tools.replacement_library import get_replacement_candidates

_llm_error_log = []
_llm_consecutive_failures = 0
_LLM_FAILURE_LIMIT = 3
_debate_call_budget = {"remaining": 100}

def set_debate_budget(n):
    """토의(debate)에 쓸 수 있는 총 LLM 호출 수 상한을 재설정."""
    _debate_call_budget["remaining"] = n

_injected_sascorer = {"module": None}
_injected_tox_predictor = {"fn": None}

def set_sascorer_module(module):
    """세션마다 다운로드/import한 sascorer 모듈을 등록."""
    _injected_sascorer["module"] = module

def set_tox_predictor(fn):
    """(original_smiles, fixed_smiles, rule_name) -> tox_delta(float) 또는 None
    을 반환하는 콜백을 등록. 노트북마다 다르게 학습한 baseline 모델을
    감싸서 넘기면 됨."""
    _injected_tox_predictor["fn"] = fn

def _call_llm(client, model_name, prompt, client_type="gemini"):
    """client_type에 따라 Gemini SDK 또는 OpenAI 호환 SDK로 호출하고,
    응답 텍스트만 통일된 형태로 반환."""
    if client_type == "gemini":
        response = client.models.generate_content(model=model_name, contents=prompt)

In [8]:
%%writefile src/tools/agent.py

import json
import time
from src.tools.replacement_library import get_replacement_candidates

_llm_error_log = []
_llm_consecutive_failures = 0
_LLM_FAILURE_LIMIT = 3
_debate_call_budget = {"remaining": 100}

def set_debate_budget(n):
    """토의(debate)에 쓸 수 있는 총 LLM 호출 수 상한을 재설정."""
    _debate_call_budget["remaining"] = n

_injected_sascorer = {"module": None}
_injected_tox_predictor = {"fn": None}

def set_sascorer_module(module):
    """세션마다 다운로드/import한 sascorer 모듈을 등록."""
    _injected_sascorer["module"] = module

def set_tox_predictor(fn):
    """(original_smiles, fixed_smiles, rule_name) -> tox_delta(float) 또는 None
    을 반환하는 콜백을 등록. 노트북마다 다르게 학습한 baseline 모델을
    감싸서 넘기면 됨."""
    _injected_tox_predictor["fn"] = fn

def _call_llm(client, model_name, prompt, client_type="gemini"):
    """client_type에 따라 Gemini SDK 또는 OpenAI 호환 SDK로 호출하고,
    응답 텍스트만 통일된 형태로 반환."""
    if client_type == "gemini":
        response = client.models.generate_content(model=model_name, contents=prompt)
        return response.text
    elif client_type == "openai_compatible":
        for attempt in range(2):  # rate limit 시 1회만 재시도
            try:
                response = client.chat.completions.create(
                    model=model_name,
                    messages=[{"role": "user", "content": prompt}],
                    max_tokens=500,
                    timeout=30,
                    extra_body={"enable_thinking": False},
                )
                return response.choices[0].message.content
            except Exception as e:
                _llm_error_log.append(repr(e))
                if 'RateLimitError' in type(e).__name__ and attempt == 0:
                    time.sleep(3)
                    continue
                return f"ERROR: LLM 호출 실패/타임아웃 - {e}"
    else:
        raise ValueError(f"알 수 없는 client_type: {client_type}")


def _parse_json_response(text, fallback):
    text = text.strip()
    if text.startswith('```'):
        text = text.split('```')[1]
        if text.startswith('json'):
            text = text[4:]
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return fallback

def _try_get_docking_evidence(rule_name, smiles_before, smiles_after):
    """도킹 표적이 등록된 규칙이면 자동으로 도킹 실행, 아니면 None.
    도킹 실패/미등록/예외는 전부 조용히 None으로 처리(critic 프롬프트에서
    도킹 근거 없이 진행하는 것으로 자연스럽게 폴백)."""
    try:
        from src.tools.docking import auto_dock_precedent, DOCKING_TARGETS
        if rule_name not in DOCKING_TARGETS:
            return None
        result = auto_dock_precedent(rule_name, smiles_before, smiles_after)
        if result.get('error') or result.get('delta') is None:
            return None
        return result
    except Exception:
        return None

def _try_compute_score(rule_name, smiles_before, smiles_after, docking_evidence=None):
    """등록된 sascorer/tox_predictor/도킹 결과를 모아 종합 점수 계산.
    일부만 등록돼 있어도 compute_multi_objective_score가 나머지로
    자동 정규화하므로 실패하지 않음. 계산 자체가 실패하면 None."""
    try:
        from src.tools.scoring import compute_multi_objective_score

        tox_delta = None
        if _injected_tox_predictor["fn"] is not None:
            try:
                tox_delta = _injected_tox_predictor["fn"](smiles_before, smiles_after, rule_name)
            except Exception:
                tox_delta = None

        precedent_docking_delta = docking_evidence["delta"] if docking_evidence else None

        return compute_multi_objective_score(
            smiles_before, smiles_after, rule_name,
            tox_delta=tox_delta,
            sascorer_module=_injected_sascorer["module"],
            precedent_docking_delta=precedent_docking_delta,
        )
    except Exception:
        return None

def _try_get_activity_risk(rule_name, smiles_before, smiles_after):
    """활성 보존 위험도 평가. sascorer가 등록 안 돼 있으면 조용히 None
    (activity_metrics가 sascorer_module을 필수로 요구하므로)."""
    try:
        if _injected_sascorer["module"] is None:
            return None
        from src.tools.activity_metrics import (compute_activity_preservation_metrics,
                                                   classify_activity_risk_v3)
        metrics = compute_activity_preservation_metrics(
            smiles_before, smiles_after, _injected_sascorer["module"]
        )
        if metrics is None:
            return None
        return classify_activity_risk_v3(metrics)
    except Exception:
        return None

def ask_llm_which_problem_to_fix(client, model_name, smiles, problems, client_type="gemini"):
    """여러 toxicophore 중 어떤 것부터 고칠지 LLM에게 판단을 요청."""
    known = [p for p in problems if get_replacement_candidates(p['rule_name']) is not None]

    if not known:
        return None
    if len(known) == 1:
        return {"rule_name": known[0]['rule_name'], "reason": "유일한 치환 가능 후보"}

    prompt = f"""당신은 신약개발 화학자입니다. 다음 분자에서 여러 구조적 문제(toxicophore)가 발견되었습니다.

분자 SMILES: {smiles}

발견된 문제 중, 우리가 실제로 치환 가능한 것들:
{json.dumps(known, ensure_ascii=False, indent=2)}

이 중 어떤 문제를 먼저 해결하는 것이 화학적으로 더 타당한지 판단하고,
반드시 아래 JSON 형식으로만 답하세요. 다른 설명 없이 JSON만 출력하세요.

{{"rule_name": "선택한 문제의 rule_name", "reason": "선택 이유 한 문장"}}
"""

    text = _call_llm(client, model_name, prompt, client_type)
    fallback = {"rule_name": known[0]['rule_name'], "reason": "JSON 파싱 실패, 기본값(첫 번째 후보) 사용"}
    return _parse_json_response(text, fallback)


def ask_llm_which_candidate_to_use(client, model_name, smiles, rule_name, client_type="gemini"):
    """한 문제(rule_name)에 대한 여러 치환 후보 중 어떤 걸 쓸지 LLM에게 판단 요청.

    candidate의 rationale 중 하나라도 '[참고]'로 시작하는 문구가 있으면,
    이는 실제 승인약물 사례에서 이 골격이 안전하게 쓰인 경우가 있다는 뜻이므로,
    candidate가 1개뿐이더라도(원래는 LLM 호출을 건너뛰던 경우) 반드시 LLM에게
    판단을 맡긴다. 이 경우 LLM은 candidate_idx로 -1을 반환하여 "치환을
    보류하고 사람(연구자) 검토가 필요하다"고 명시적으로 표시할 수 있다.
    """
    info = get_replacement_candidates(rule_name)
    if info is None:
        return None

    candidates = info['candidates']
    has_caution = any('[참고]' in c.get('rationale', '') for c in candidates)

    if len(candidates) == 1 and not has_caution:
        return {"candidate_idx": 0, "reason": "유일한 후보"}

    candidate_info = [
        {"idx": i, "name": c['name'], "rationale": c['rationale']}
        for i, c in enumerate(candidates)
    ]

    prompt = f"""당신은 신약개발 화학자입니다. 다음 분자에서 '{rule_name}' 문제를
해결하기 위한 치환 후보가 있습니다.

분자 SMILES: {smiles}

치환 후보들:
{json.dumps(candidate_info, ensure_ascii=False, indent=2)}

각 후보의 rationale에 "[참고]"로 시작하는 문구가 있다면, 이는 "이 골격이
실제 승인 약물에서 반응성이 아닌 안정적 형태로 널리 쓰인 사례가 있으니,
경고를 절대적 기준이 아닌 참고 신호로 해석하라"는 뜻입니다. 이 경우 먼저
"이 분자가 그 참고사항이 가리키는 안전한 사용 사례와 실제로 유사한지"를
판단하세요.
- 유사하다고 판단되면서, 후보가 여러 개라면 변화 폭이 더 작은 후보를 선택하세요.
- 유사하다고 판단되고, 치환 자체가 불필요하다고 볼 만큼 뚜렷하다면,
  candidate_idx를 -1로 답해 "치환 보류, 사람 검토 필요"를 표시하세요.
- 참고사항이 없거나 이 분자가 그 사례와 유사하지 않다면, 평소대로 가장
  적절한 후보를 선택하세요.

반드시 아래 JSON 형식으로만 답하세요. 다른 설명 없이 JSON만 출력하세요.

{{"candidate_idx": 선택한 후보의 idx(정수, 또는 보류 시 -1), "reason": "판단 이유 한 문장"}}
"""

    text = _call_llm(client, model_name, prompt, client_type)
    fallback = {"candidate_idx": 0, "reason": "JSON 파싱 실패, 기본값(첫 번째 후보) 사용"}
    result = _parse_json_response(text, fallback)

    idx = result.get('candidate_idx')
    if not isinstance(idx, int) or not (-1 <= idx < len(candidates)):
        return {"candidate_idx": 0, "reason": "LLM 응답 idx 범위 오류, 기본값 사용"}
    return result


def ask_llm_debate_fix(client, model_name, smiles_before, smiles_after, rule_name,
                        candidate_name, candidate_rationale, client_type="gemini",
                        max_rounds=2):
    """제안자(원래 candidate를 고른 논리)와 검토자(critic)가 여러 라운드
    대화하며 합의에 도달하려 시도. 매 라운드 critic이 판단하고, 반려하면
    proposer가 반박, critic이 재판단. max_rounds 안에 합의(양쪽 다 승인,
    또는 critic이 최종 반려로 확정) 안 되면 "escalate"로 사람 검토行.

    반환: {"final_verdict": "approved"|"rejected"|"escalate",
           "rounds": [{"role": "critic"|"proposer", "text": str}, ...],
           "consensus_reached": bool}
    """
    if _debate_call_budget["remaining"] <= 0:
        return {"final_verdict": "approved", "rounds": [], "consensus_reached": True,
                "budget_exhausted": True}
    _debate_call_budget["remaining"] -= 1
    rounds_log = []
    proposer_argument = candidate_rationale

    for round_num in range(1, max_rounds + 1):
        docking_evidence = _try_get_docking_evidence(rule_name, smiles_before, smiles_after)
        score_result = _try_compute_score(rule_name, smiles_before, smiles_after, docking_evidence)
        activity_risk = _try_get_activity_risk(rule_name, smiles_before, smiles_after)
        critic_prompt = f"""당신은 신약개발 화학 검토자(critic)입니다. 동료 화학자가 아래
치환을 제안했습니다.

원본 분자: {smiles_before}
치환 후 분자: {smiles_after}
해결하려던 문제: {rule_name}
제안된 치환: {candidate_name}
제안자의 근거: {proposer_argument}
{f"실측 도킹 결합력 변화: {docking_evidence['target']} 표적, {docking_evidence['score_original']:.2f} → {docking_evidence['score_fixed']:.2f} kcal/mol (delta {docking_evidence['delta']:+.2f}). 이 정량 데이터를 판단에 반영하세요." if docking_evidence else ""}
{f"종합 점수: {score_result['composite_score']:.2f} (세부: {score_result['component_scores']}). 이것도 판단에 참고하세요." if score_result and score_result.get('composite_score') is not None else ""}
{f"활성 보존 위험도 평가: {activity_risk['verdict']} (세부: {'; '.join(activity_risk['details'])})" if activity_risk else ""}

이 치환에 동의하는지 비판적으로 검토하세요. 동의하지 않는다면 구체적으로
어떤 점이 문제인지 명시하세요(새로운 독성 구조 생성 가능성, 근거의
논리적 결함, precedent 오독 등).

반드시 아래 JSON 형식으로만 답하세요.
{{"verdict": "approved" 또는 "rejected", "reason": "판단 이유, 반려 시 구체적 반론 포함"}}
"""
        critic_text = _call_llm(client, model_name, critic_prompt, client_type)
        critic_result = _parse_json_response(
            critic_text, {"verdict": "approved", "reason": "JSON 파싱 실패, 기본 승인"}
        )
        rounds_log.append({"role": "critic", "round": round_num, "text": critic_result})

        if critic_result.get("verdict") == "approved":
            return {"final_verdict": "approved", "rounds": rounds_log, "consensus_reached": True}

        if round_num == max_rounds:
            break

        proposer_prompt = f"""당신은 방금 아래 치환을 제안한 화학자입니다.

원본 분자: {smiles_before}
치환 후 분자: {smiles_after}
당신의 원래 근거: {proposer_argument}

동료 검토자(critic)가 다음과 같이 반려했습니다: "{critic_result.get('reason', '')}"

이 반론에 대해 답하세요. 반론이 타당하면 인정하고 제안을 철회하세요.
반론이 부당하다면 왜 원래 치환이 여전히 타당한지 반박하세요.

반드시 아래 JSON 형식으로만 답하세요.
{{"stance": "withdraw" 또는 "defend", "argument": "반박 또는 철회 이유"}}
"""
        proposer_text = _call_llm(client, model_name, proposer_prompt, client_type)
        proposer_result = _parse_json_response(
            proposer_text, {"stance": "withdraw", "argument": "JSON 파싱 실패, 기본 철회"}
        )
        rounds_log.append({"role": "proposer", "round": round_num, "text": proposer_result})

        if proposer_result.get("stance") == "withdraw":
            return {"final_verdict": "rejected", "rounds": rounds_log, "consensus_reached": True}

        proposer_argument = proposer_result.get("argument", proposer_argument)

    return {"final_verdict": "escalate", "rounds": rounds_log, "consensus_reached": False}
def should_debate(candidate_rationale):
    """이 candidate가 토의(debate)를 거칠 필요가 있는지 판단.
    rationale에 '[참고]'가 있으면 실제 승인약물 사례와 겹칠 수 있다는
    뜻이므로, 단순 채택 대신 토의로 한 번 더 검토해야 함."""
    return '[참고]' in (candidate_rationale or '')


Overwriting src/tools/agent.py


In [12]:
class RecordingMockClient:
    class _Choice:
        def __init__(self, content):
            self.message = type('obj', (), {'content': content})
    class _Response:
        def __init__(self, content):
            self.choices = [RecordingMockClient._Choice(content)]
    class _Completions:
        def __init__(self, canned):
            self.canned = canned
            self.log = []
        def create(self, **kwargs):
            self.log.append(kwargs['messages'][0]['content'])
            return RecordingMockClient._Response(self.canned)
    class _Chat:
        def __init__(self, canned):
            self.completions = RecordingMockClient._Completions(canned)
    def __init__(self, canned):
        self.chat = RecordingMockClient._Chat(canned)

In [13]:
tox_model = train_tox21_baseline(data)
tox_predictor = make_tox_predictor(tox_model)

from src.tools.agent import set_sascorer_module, set_tox_predictor
set_sascorer_module(sascorer)
set_tox_predictor(tox_predictor)

mock = RecordingMockClient('{"verdict": "approved", "reason": "타당함"}')
r = ask_llm_debate_fix(
    mock, "mock", "NCCc1ccc(O)c(O)c1", "COc1ccc(CCN)cc1O", "catechol",
    "methylated catechol", "[참고] 카테콜 골격은...", client_type="openai_compatible",
)
log = mock.chat.completions.log
print(log[0])

당신은 신약개발 화학 검토자(critic)입니다. 동료 화학자가 아래
치환을 제안했습니다.

원본 분자: NCCc1ccc(O)c(O)c1
치환 후 분자: COc1ccc(CCN)cc1O
해결하려던 문제: catechol
제안된 치환: methylated catechol
제안자의 근거: [참고] 카테콜 골격은...
실측 도킹 결합력 변화: COMT 표적, -5.18 → -5.19 kcal/mol (delta -0.01). 이 정량 데이터를 판단에 반영하세요.
종합 점수: 0.66 (세부: {'toxicity': 0.53, 'docking': 0.5050000000000003, 'sa': 1.0, 'qed': 0.6580250816015243, 'lipinski': 0.5, 'pains': 1.0}). 이것도 판단에 참고하세요.
활성 보존 위험도 평가: 구조·형태 모두 보존 — 활성 유지 가능성 높음 [보조 경고: QED 변화] (세부: 2D 연결성: 유사 (Tanimoto 0.600); 3D 형태: 보존 (회전반경 +6.8%); 약물유사성(QED): 변화 (+0.158); 소수성(LogP): 유지 (+0.303); 합성용이성(SA): 유지/개선 (-0.147))

이 치환에 동의하는지 비판적으로 검토하세요. 동의하지 않는다면 구체적으로
어떤 점이 문제인지 명시하세요(새로운 독성 구조 생성 가능성, 근거의
논리적 결함, precedent 오독 등).

반드시 아래 JSON 형식으로만 답하세요.
{"verdict": "approved" 또는 "rejected", "reason": "판단 이유, 반려 시 구체적 반론 포함"}



In [14]:
import src.tools.activity_metrics
importlib.reload(src.tools.activity_metrics)
importlib.reload(src.tools.agent)
from src.tools.agent import ask_llm_debate_fix, set_sascorer_module, set_tox_predictor

set_sascorer_module(sascorer)
set_tox_predictor(tox_predictor)

mock = RecordingMockClient('{"verdict": "approved", "reason": "타당함"}')
r = ask_llm_debate_fix(
    mock, "mock", "NCCc1ccc(O)c(O)c1", "COc1ccc(CCN)cc1O", "catechol",
    "methylated catechol", "[참고] 카테콜 골격은...", client_type="openai_compatible",
)
log = mock.chat.completions.log
print(log[0])

당신은 신약개발 화학 검토자(critic)입니다. 동료 화학자가 아래
치환을 제안했습니다.

원본 분자: NCCc1ccc(O)c(O)c1
치환 후 분자: COc1ccc(CCN)cc1O
해결하려던 문제: catechol
제안된 치환: methylated catechol
제안자의 근거: [참고] 카테콜 골격은...
실측 도킹 결합력 변화: COMT 표적, -5.18 → -5.19 kcal/mol (delta -0.01). 이 정량 데이터를 판단에 반영하세요.
종합 점수: 0.66 (세부: {'toxicity': 0.53, 'docking': 0.5050000000000003, 'sa': 1.0, 'qed': 0.6580250816015243, 'lipinski': 0.5, 'pains': 1.0}). 이것도 판단에 참고하세요.
활성 보존 위험도 평가: 구조·형태 모두 보존 — 활성 유지 가능성 높음 [보조 경고: QED 변화] (세부: 2D 연결성: 유사 (Tanimoto 0.600); 3D 형태: 보존 (회전반경 +6.8%); 약물유사성(QED): 변화 (+0.158); 소수성(LogP): 유지 (+0.303); 합성용이성(SA): 유지/개선 (-0.147))

이 치환에 동의하는지 비판적으로 검토하세요. 동의하지 않는다면 구체적으로
어떤 점이 문제인지 명시하세요(새로운 독성 구조 생성 가능성, 근거의
논리적 결함, precedent 오독 등).

반드시 아래 JSON 형식으로만 답하세요.
{"verdict": "approved" 또는 "rejected", "reason": "판단 이유, 반려 시 구체적 반론 포함"}



In [15]:
import ast
with open('src/tools/agent.py') as f:
    content = f.read()
    ast.parse(content)
print("✅ 문법 정상")
print("✅ _try_get_activity_risk 반영:", '_try_get_activity_risk' in content)

importlib.reload(src.tools.activity_metrics)
importlib.reload(src.tools.agent)
from src.tools.agent import ask_llm_debate_fix, set_sascorer_module, set_tox_predictor

# 이 세션의 sascorer/tox_predictor 재등록 (reload로 초기화됐을 수 있음)
set_sascorer_module(sascorer)
set_tox_predictor(tox_predictor)

mock = RecordingMockClient('{"verdict": "approved", "reason": "타당함"}')
r = ask_llm_debate_fix(
    mock, "mock", "NCCc1ccc(O)c(O)c1", "COc1ccc(CCN)cc1O", "catechol",
    "methylated catechol", "[참고] 카테콜 골격은...", client_type="openai_compatible",
)
log = mock.chat.completions.log
print(log[0])

✅ 문법 정상
✅ _try_get_activity_risk 반영: True
당신은 신약개발 화학 검토자(critic)입니다. 동료 화학자가 아래
치환을 제안했습니다.

원본 분자: NCCc1ccc(O)c(O)c1
치환 후 분자: COc1ccc(CCN)cc1O
해결하려던 문제: catechol
제안된 치환: methylated catechol
제안자의 근거: [참고] 카테콜 골격은...
실측 도킹 결합력 변화: COMT 표적, -5.18 → -5.19 kcal/mol (delta -0.01). 이 정량 데이터를 판단에 반영하세요.
종합 점수: 0.66 (세부: {'toxicity': 0.53, 'docking': 0.5050000000000003, 'sa': 1.0, 'qed': 0.6580250816015243, 'lipinski': 0.5, 'pains': 1.0}). 이것도 판단에 참고하세요.
활성 보존 위험도 평가: 구조·형태 모두 보존 — 활성 유지 가능성 높음 [보조 경고: QED 변화] (세부: 2D 연결성: 유사 (Tanimoto 0.600); 3D 형태: 보존 (회전반경 +6.8%); 약물유사성(QED): 변화 (+0.158); 소수성(LogP): 유지 (+0.303); 합성용이성(SA): 유지/개선 (-0.147))

이 치환에 동의하는지 비판적으로 검토하세요. 동의하지 않는다면 구체적으로
어떤 점이 문제인지 명시하세요(새로운 독성 구조 생성 가능성, 근거의
논리적 결함, precedent 오독 등).

반드시 아래 JSON 형식으로만 답하세요.
{"verdict": "approved" 또는 "rejected", "reason": "판단 이유, 반려 시 구체적 반론 포함"}



In [16]:
!cd /content/laidd-2026 && git add -A && git commit -m "Connect activity preservation risk assessment into debate critic prompt via _try_get_activity_risk" && git push

[main c141d87] Connect activity preservation risk assessment into debate critic prompt via _try_get_activity_risk
 1 file changed, 20 insertions(+)
Enumerating objects: 9, done.
Counting objects: 100% (9/9), done.
Delta compression using up to 2 threads
Compressing objects: 100% (5/5), done.
Writing objects: 100% (5/5), 875 bytes | 875.00 KiB/s, done.
Total 5 (delta 3), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (3/3), completed with 3 local objects.
To https://github.com/Dec32th/laidd-2026.git
   4c20dce..c141d87  main -> main


In [17]:
smi = "NNC(=O)CP(=O)(c1ccccc1)c1ccccc1"
info = get_replacement_candidates('phosphor')
print("problem_smarts:", info['problem_smarts'])
for i, c in enumerate(info['candidates']):
    print(f"\ncandidate idx={i}")
    for k, v in c.items():
        print(f"  {k}: {v}")

# 이 분자에서 phosphor 규칙이 실제로 매치되는지, edit_type이 뭘 하려는지 확인
from rdkit import Chem
pattern = Chem.MolFromSmarts(info['problem_smarts'])
mol = Chem.MolFromSmiles("NC(=O)CP(=O)(c1ccccc1)c1ccccc1")  # hydrazine 고친 후 상태
matches = mol.GetSubstructMatches(pattern)
print("\nphosphor 매치:", matches)
for m in matches:
    for idx in m:
        atom = mol.GetAtomWithIdx(idx)
        print(f"  atom {idx}: {atom.GetSymbol()}, degree={atom.GetDegree()}, neighbors={[n.GetSymbol() for n in atom.GetNeighbors()]}")

problem_smarts: [OX2][PX4](=[OX1])([OX2])[OX2]

candidate idx=0
  edit_type: cleave_bond
  cleave_pair_in_pattern: (1, 4)
  name: diester + phenol/alcohol (one ester bond cleaved)
  rationale: 유기인산 트리에스터(트리아릴/트리알킬 포스페이트)는 아세틸콜린에스터라제(AChE) 억제를 통한 신경독성 메커니즘이 잘 알려진 구조로(유기인계 살충제·신경작용제의 공통 골격), 다중 에스터 결합이 반응성/생체이용률에 기여함. 에스터 결합 하나를 가수분해로 끊어 반응성을 낮춤 (검증 필요, 인 원자에 남은 나머지 에스터는 추가 규칙 필요 가능)

phosphor 매치: ()


In [18]:
count_ester = 0
count_non_ester = 0
for smi in data['smiles_valid']:
    problems = detect_toxicophores(smi)
    if any(p['rule_name'] == 'phosphor' for p in problems):
        mol = Chem.MolFromSmiles(smi)
        if mol.HasSubstructMatch(pattern):
            count_ester += 1
        else:
            count_non_ester += 1

print(f"phosphor 중 에스터형(현재 처리 가능): {count_ester}")
print(f"phosphor 중 비에스터형(포스핀옥사이드 등, 처리 불가): {count_non_ester}")

phosphor 중 에스터형(현재 처리 가능): 9
phosphor 중 비에스터형(포스핀옥사이드 등, 처리 불가): 24


In [20]:
from itertools import islice

query = "O=P(c1ccccc1)(c1ccccc1)c1ccccc1"  # 트리페닐포스핀옥사이드 코어

hits = list(islice(new_client.substructure.filter(smiles=query), 100))
print(f"서브구조 매치: {len(hits)}건")

approved = []
for h in hits:
    cid = h['molecule_chembl_id']
    res = list(molecule.filter(molecule_chembl_id=cid, max_phase=4))
    if res:
        rec = res[0]
        smi = rec.get('molecule_structures', {}).get('canonical_smiles') if rec.get('molecule_structures') else None
        approved.append((cid, rec.get('pref_name'), rec.get('withdrawn_flag'), smi))

print(f"\n승인(phase4) 약물: {len(approved)}건")
for a in approved:
    print(a)

서브구조 매치: 62건

승인(phase4) 약물: 0건


In [21]:
!cat src/tools/precedent_library.py


"""선례 라이브러리 — 승인/철수 약물, 정량 활성 데이터, 도킹 검증 결과를
판단 에이전트 프롬프트에 실시간 주입하기 위한 구조화된 근거 저장소.
모든 항목은 이 세션에서 ChEMBL/GtoPdb API 조회 또는 실제 도킹 실행으로
직접 확인한 것만 포함한다(추정/일반 지식은 배제).
"""

PRECEDENT_LIBRARY = [
    {"rule": "Thiocarbonyl_group", "type": "긍정_승인약물쌍",
     "description": "티오펜탈(C=S)/펜토바비탈(C=O), 티아밀랄(C=S)/세코바비탈(C=O) - "
                     "동일 사이드체인, C=S->C=O만 다른 실제 승인 마취제 쌍. "
                     "baseline 모델 기준 옥소형이 티오형보다 Tox21 평균 예측값 낮음(-0.008~-0.009)."},
    {"rule": "catechol", "type": "정량_활성데이터",
     "description": "도파민이 D1(Ki 4.3-5.6nM)/D2(Ki 4.7-7.2nM)/D3(Ki 6.4-7.3nM) 수용체에 "
                     "단자릿수 nM 강력 결합 - 카테콜 골격이 활성에 필수적임을 정량적으로 뒷받침."},
    {"rule": "hydroxamic_acid", "type": "부정_참고사례_검증필요",
     "description": "하이드록삼산 골격(보리노스타트 등 HDAC 억제제)은 아연 킬레이션이 "
                     "약효 핵심이므로, 이 계열에 대한 무분별한 치환은 약효 상실 위험. "
                     "(문헌 재확인 필요)"},
    {"rule": "beta-keto/anhydride", "type": "긍정_통계검증결과",
     "description": "MMPDB 공식 통계 도구로 재검증한 결과, Tox21 규모(1173개)에서 "
    

In [22]:
%%writefile src/tools/precedent_library.py

"""선례 라이브러리 — 승인/철수 약물, 정량 활성 데이터, 도킹 검증 결과를
판단 에이전트 프롬프트에 실시간 주입하기 위한 구조화된 근거 저장소.
모든 항목은 이 세션에서 ChEMBL/GtoPdb API 조회 또는 실제 도킹 실행으로
직접 확인한 것만 포함한다(추정/일반 지식은 배제).
"""

PRECEDENT_LIBRARY = [
    {"rule": "Thiocarbonyl_group", "type": "긍정_승인약물쌍",
     "description": "티오펜탈(C=S)/펜토바비탈(C=O), 티아밀랄(C=S)/세코바비탈(C=O) - "
                     "동일 사이드체인, C=S->C=O만 다른 실제 승인 마취제 쌍. "
                     "baseline 모델 기준 옥소형이 티오형보다 Tox21 평균 예측값 낮음(-0.008~-0.009)."},
    {"rule": "catechol", "type": "정량_활성데이터",
     "description": "도파민이 D1(Ki 4.3-5.6nM)/D2(Ki 4.7-7.2nM)/D3(Ki 6.4-7.3nM) 수용체에 "
                     "단자릿수 nM 강력 결합 - 카테콜 골격이 활성에 필수적임을 정량적으로 뒷받침."},
    {"rule": "hydroxamic_acid", "type": "부정_참고사례_검증필요",
     "description": "하이드록삼산 골격(보리노스타트 등 HDAC 억제제)은 아연 킬레이션이 "
                     "약효 핵심이므로, 이 계열에 대한 무분별한 치환은 약효 상실 위험. "
                     "(문헌 재확인 필요)"},
    {"rule": "beta-keto/anhydride", "type": "긍정_통계검증결과",
     "description": "MMPDB 공식 통계 도구로 재검증한 결과, Tox21 규모(1173개)에서 "
                     "무수물 관련 매칭쌍은 표본 부족(count=1)으로 통계적 유의성 확보 불가 - "
                     "데이터형 접근보다 문헌형 근거가 더 신뢰할 만함을 시사."},
    {"rule": "Michael_acceptor_1", "type": "위험=메커니즘_참고",
     "description": "에타크린산(이뇨제, FDA 승인)은 시스테인 잔기와의 공유결합 자체가 "
                     "작용 메커니즘인 공유결합 억제제 - Michael acceptor 경고가 항상 "
                     "제거 대상은 아님을 보여주는 실제 승인약물 사례."},
    {"rule": "alkyl_halide", "type": "위험=메커니즘_참고",
     "description": "메클로르에타민, 사이클로포스파미드 등 알킬화 항암제는 DNA 알킬화 "
                     "반응성 자체가 세포독성 치료 메커니즘 - 이 계열에는 할로겐 제거가 "
                     "부적절함을 보여주는 실제 승인약물 사례."},
    {"rule": "azo_A(324)", "type": "위험=메커니즘_참고_검증완료",
     "description": "설파살라진(SMILES 내 /N=N/ 아조 결합 확인, ChEMBL max_phase=4.0, "
                     "GtoPdb FDA 승인 1950년/WHO 필수의약품)은 아조 결합이 장내 "
                     "세균에 의해 환원되어 활성 대사물(5-ASA)을 방출하는 프로드러그 - "
                     "실제 조회로 검증됨."},
    {"rule": "catechol", "type": "도킹검증_결과",
     "description": "COMT(PDB 1VID) 도킹 검증: 도파민(-5.72 kcal/mol)→메톡시도파민"
                     "(-5.41 kcal/mol), 변화폭 +0.31 kcal/mol로 약화 방향이나 이는 "
                     "1 kcal/mol 미만의 작은 차이로 도킹 자체의 오차범위 내일 수 있어 "
                     "단정적 근거로 삼기엔 약함. 에피네프린은 반대로 미세 강화"
                     "(-6.21→-6.32, -0.10) - 두 경우 모두 변화폭이 작아, 도킹 수치보다는 "
                     "카테콜의 수용체 결합 필수성(정성적 근거)이 더 강한 판단 기준."},
    {"rule": "Michael_acceptor_1", "type": "도킹검증_방법론한계",
     "description": "EGFR(PDB 6JX4) 도킹 검증: 오시메르티닙(-7.13)→C=C환원버전(-7.08), "
                     "거의 무변화(+0.05). 표준(비공유) 도킹이 오시메르티닙의 실제 "
                     "공유결합(Cys797) 메커니즘을 포착하지 못하는 방법론적 한계 확인 - "
                     "공유결합 억제제 계열은 일반 도킹 스코어만으로 활성 손실을 판단하지 "
                     "말 것(도킹 무변화가 곧 활성 유지를 뜻하지 않음)."},
    {"rule": "hydroquinone", "type": "도킹검증_결과",
     "description": "NQO1 도킹 검증: 퀴논(-3.29)→하이드로퀴논(-4.08), 결합 강화(-0.79, "
                     "1 kcal/mol에 근접하는 뚜렷한 변화). 메틸퀴논(-3.80)→환원버전"
                     "(-4.29)도 강화(-0.49), 2건 모두 일관되게 강화 방향. NQO1이 실제로 "
                     "퀴논을 하이드로퀴논으로 환원하는 효소이므로, 이 치환 방향은 해독 "
                     "반응경로와 자연스럽게 정렬되며 실측 결합력도 개선됨 - 활성 손실 "
                     "우려가 낮은 것으로 확인됨."},
    {"rule": "quinone_A(370)", "type": "도킹검증_결과",
     "description": "NQO1 도킹 검증: 퀴논(-3.29)→하이드로퀴논(-4.08), 결합 강화(-0.79). "
                     "실제 표적 효소와의 결합력이 오히려 개선되는 것으로 실측 확인됨 "
                     "(hydroquinone 규칙과 동일 표적 데이터 공유)."},
    {"rule": "Aliphatic_long_chain", "type": "긍정_승인약물_확인(구조는_동의어로_대체확인)",
     "description": "POLIDOCANOL(라우릴알코올+에틸렌옥사이드 평균 9개 반복부가체)은 ChEMBL 조회로 "
                     "승인 확인됨(max_phase=4.0, first_approval=2010, ATC C05BB02, "
                     "dosed_ingredient=True, withdrawn=False, 상품명 Asclera/Aethoxysklerol). "
                     "ChEMBL에 단일 SMILES는 없으나(polymer_flag=1, structure_type=NONE) "
                     "이는 다분산 고분자라 원천적으로 단일 구조가 없기 때문이며, 공식 동의어"
                     "(USP: Polyoxyl 9 lauryl ether, JAN: Lauromacrogol 400)가 "
                     "\"장쇄 알킬+반복 에테르\" 구조를 명확히 정의함 - 실제 승인약물에서 "
                     "이 전략이 쓰이고 있음을 뒷받침."},
    {"rule": "Aliphatic_long_chain", "type": "부정_참고사례_검증필요",
     "description": "ChEMBL 서브구조 검색(에테르 삽입 사슬 모티프)으로 매치된 승인약물은 "
                     "에리스로마이신/아지스로마이신/암포테리신B였으나, 매치 위치를 IsInRing으로 "
                     "확인한 결과 전부 매크로락톤/당 고리 내부의 고리형 에테르로, 우리 규칙이 "
                     "다루는 \"고리 밖 열린 사슬\" 상황과는 구조적으로 다름 - 이 계열은 직접적 "
                     "근거로 부적합함이 확인됨."},
    {"rule": "isolated_alkene", "type": "긍정_승인약물쌍",
     "description": "SIROLIMUS(시롤리무스), TACROLIMUS ANHYDROUS(타크로리무스) - 둘 다 ChEMBL "
                     "조회로 승인·비철수 확인됨(withdrawn_flag=False), 대형 매크로라이드 면역억제제로 "
                     "현재도 널리 처방됨. problem_smarts로 직접 매치되는 고립 지방족 알켄이 구조 "
                     "안에 실제 존재 - 고립 알켄이 항상 제거 대상은 아님을 보여주는 실제 승인약물 사례."},
    {"rule": "isolated_alkene", "type": "위험=메커니즘_참고_인과불명",
     "description": "CYCLOBARBITAL, HEXOBARBITAL 둘 다 ChEMBL 조회로 withdrawn_flag=True 확인됨, "
                     "둘 다 problem_smarts에 매치되는 사이클로헥세닐 고립 알켄 치환기를 가짐. "
                     "다만 바르비투르산염 계열은 호흡억제·의존성 등 일반적 안전성 문제로 철수된 "
                     "사례가 많아, 이 알켄 구조가 철수의 직접 원인이라는 인과관계는 확인되지 않음 "
                     "(상관관계만 관찰, 문헌 추가 확인 필요)."},
    {"rule": "nitro_group", "type": "위험=메커니즘_참고_검증완료",
     "description": "METRONIDAZOLE, NITROFURANTOIN, BENZNIDAZOLE 셋 다 ChEMBL 조회로 승인·비철수 "
                     "확인됨(max_phase=4.0, withdrawn_flag=False), SMILES에 니트로기([N+](=O)[O-]) "
                     "실제 존재 확인. 항균/항기생충제 계열에서 니트로기의 선택적 환원 활성화 자체가 "
                     "치료 메커니즘인 프로드러그 설계 사례 - 이런 계열에는 니트로기 제거가 "
                     "부적절함을 실제 조회로 검증함."},
    {"rule": "aniline", "type": "위험=메커니즘_참고_검증완료",
     "description": "SULFANILAMIDE, SULFAMETHOXAZOLE, PROCAINAMIDE 셋 다 ChEMBL 조회로 승인·비철수 "
                     "확인됨(max_phase=4.0, withdrawn_flag=False), SMILES 확인 결과 셋 다 아실화되지 "
                     "않은 유리 1차 방향족 아민(아닐린) 형태로 실제 처방됨. 설파계 항생제·항부정맥제 "
                     "계열에서 특이체질 반응 위험에도 불구하고 유리 아닐린 골격이 오랜 기간 널리 "
                     "쓰여온 사례 - 이 경고가 절대적 배제 기준이 아님을 실제 조회로 검증함."},
    {"rule": "Sulfonic_acid_2", "type": "위험=메커니즘_참고_검증완료",
     "description": "LISDEXAMFETAMINE DIMESYLATE, SAQUINAVIR MESYLATE 둘 다 ChEMBL 조회로 승인·비철수 "
                     "확인됨(max_phase=4.0, withdrawn_flag=False). RDKit GetMolFrags로 분자 조각을 "
                     "분리해 확인한 결과, 설폰산(메실산) 매치는 둘 다 작은 카운터이온 조각(CS(=O)(=O)O, "
                     "5원자)에서만 나오고 주 약효 골격 조각(19원자, 49원자)에서는 전혀 매치되지 않음 - "
                     "설폰산이 활성 골격이 아니라 순수 염 형성용 카운터이온인 경우가 실제로 존재함을 "
                     "구조적으로 검증함. 이런 경우 본 규칙의 치환 대상이 아님."},
    {"rule": "phosphor", "type": "위험=메커니즘_참고_검증완료",
     "description": "FOSPHENYTOIN(유리산 형태) ChEMBL 조회로 승인·비철수 확인됨(max_phase=4.0, "
                     "withdrawn_flag=False), problem_smarts 실제 매치 확인. 페니토인의 인산에스터 "
                     "프로드러그로, 체내 인산가수분해효소에 의한 에스터 절단 자체가 설계된 방출 "
                     "메커니즘 - 이 경우 인산에스터 절단이 규칙이 우려하는 신경독성 반응성이 아니라 "
                     "오히려 활성화 경로이므로, 프로드러그 맥락에서는 본 규칙의 무분별한 적용이 "
                     "부적절할 수 있음을 실제 조회로 검증함."},
    {"rule": "phosphor", "type": "부정_참고사례_검증필요",
     "description": "phosphor 규칙의 유일한 candidate(cleave_bond)는 에스터형 산소가 있는 "
                     "유기인산 트리에스터(포스페이트)만 처리 가능함. valid set에서 phosphor로 "
                     "진단된 33건 중 24건(73%)은 에스터 산소가 없는 트리아릴포스핀옥사이드 등 "
                     "구조적으로 다른 화학종. ChEMBL 서브구조 검색(트리페닐포스핀옥사이드 코어, "
                     "62건 매치)에서 승인약물(max_phase=4) 0건 확인 — 이 골격은 약물 후보로 "
                     "개발된 적이 거의 없는(유기합성 시약/부산물에 가까운) 계열로 보이며, 안전성 "
                     "근거도 위험성 근거도 뚜렷하지 않아 새 치환 전략을 만들 화학적 정당성이 "
                     "부족함. 이 서브클래스는 stuck이 정상적 결과로 남을 수 있음(범위 밖으로 "
                     "명시, 억지 치환 candidate 추가는 지양)."},
    {"rule": "quaternary_nitrogen_1", "type": "위험=메커니즘_참고_검증완료",
     "description": "PRALIDOXIME, PRALIDOXIME CHLORIDE 둘 다 ChEMBL 조회로 승인·비철수 확인됨"
                     "(max_phase=4.0, withdrawn_flag=False), problem_smarts 실제 매치 확인"
                     "(N-메틸피리디늄 옥심). 유기인계(신경작용제/살충제) 중독 해독제로, "
                     "4차 피리디늄의 양전하 자체가 콜린에스터라제 활성부위의 음이온 결합자리를 "
                     "표적하는 데 필수적인 활성 메커니즘 - 이 계열에는 4차 질소 제거가 약효 "
                     "상실로 직결됨을 실제 조회로 검증함."},
    {"rule": "aldehyde", "type": "긍정_통계검증결과",
     "description": "ChEMBL 서브구조 검색(CCC=O, CC(C)C=O, c1ccccc1C=O, CCCCC=O, OCC=O 등 5개 쿼리, "
                     "쿼리당 최대 100건)으로 승인약물(max_phase=4) 후보 10건을 얻었으나, "
                     "problem_smarts([CX3H1](=O))로 재확인한 결과 전부 실제로는 알데히드가 아닌 "
                     "다른 카르보닐(락탐/케토락톤/에스터 등)로 밝혀짐 - 진짜 유리 알데히드를 가진 "
                     "승인약물은 0건. 알데히드의 친전자성 반응 우려가 실제로 승인 단계에서 "
                     "강하게 작용해 최종 약물 형태로 잘 남지 않음을 시사함 (반증 근거 부재 = "
                     "규칙의 타당성을 간접적으로 뒷받침)."},
    {"rule": "quaternary_nitrogen_2", "type": "위험=메커니즘_참고_검증완료",
     "description": "SUXAMETHONIUM(석시닐콜린), TUBOCURARINE, VECURONIUM, ROCURONIUM 넷 다 ChEMBL "
                     "조회로 승인·비철수 확인됨(max_phase=4.0, withdrawn_flag=False), problem_smarts "
                     "실제 매치 확인. 전부 신경근 차단제(근이완제) 계열로, 4차 암모늄의 영구 양전하가 "
                     "니코틴성 아세틸콜린 수용체 결합에 필수적인 활성 메커니즘 그 자체 - 이 계열에는 "
                     "4차 질소 제거가 약효 상실로 직결됨을 실제 조회로 검증함."},
    {"rule": "imine_1_general", "type": "위험=메커니즘_참고_검증완료",
     "description": "DIAZEPAM, CLONAZEPAM, NITRAZEPAM, OXAZEPAM, LORAZEPAM, BROMAZEPAM, "
                     "CHLORDIAZEPOXIDE, CLOZAPINE, GEMIFLOXACIN 9종 ChEMBL 조회로 승인·비철수 확인됨"
                     "(max_phase=4.0, withdrawn_flag=False), problem_smarts 실제 매치 확인. 다수가 "
                     "벤조디아제핀 계열로 역사상 가장 널리 처방된 약물군 중 하나 - 다만 이들의 C=N은 "
                     "7원 diazepine 고리 안에 갇힌 고리형 이민으로, 개방 사슬형(비고리) 쉬프 염기보다 "
                     "가수분해에 안정적인 구조적 특성이 있어 '고리형 이민'에 한정된 근거로 해석해야 함. "
                     "FLUNITRAZEPAM은 withdrawn_flag=True(남용/규제 이슈 가능성, 독성 인과 불명)."},
]


def get_precedents(rule_name: str) -> str | None:
    """규칙 이름으로 관련 선례를 찾아 프롬프트에 넣을 텍스트로 반환."""
    matches = [p for p in PRECEDENT_LIBRARY if p['rule'] == rule_name]
    if not matches:
        return None
    return "\n".join([f"- [{m['type']}] {m['description']}" for m in matches])


Overwriting src/tools/precedent_library.py


In [23]:
importlib.reload(src.tools.replacement_library)
importlib.reload(src.tools.atom_editor)
importlib.reload(src.tools.molecule_editor)
importlib.reload(src.tools.precedent_library)
from src.tools.molecule_editor import iterative_fix_loop, clear_failure_memory
from src.tools.precedent_library import PRECEDENT_LIBRARY, get_precedents
clear_failure_memory()

import ast
with open('src/tools/precedent_library.py') as f:
    ast.parse(f.read())
print("✅ 문법 정상")

assert len(PRECEDENT_LIBRARY) == 24, f"개수 불일치: {len(PRECEDENT_LIBRARY)}"
print(f"✅ 총 {len(PRECEDENT_LIBRARY)}건")

print(get_precedents('phosphor'))

smoke_result = iterative_fix_loop('CCCCCCCCCCCCCCCC', max_iterations=10, candidate_idx=0)
assert smoke_result['status'] == 'success'
print("✅ 스모크 테스트 통과")
print("\n전체 통과 — 커밋해도 안전합니다.")

✅ 문법 정상
✅ 총 24건
- [위험=메커니즘_참고_검증완료] FOSPHENYTOIN(유리산 형태) ChEMBL 조회로 승인·비철수 확인됨(max_phase=4.0, withdrawn_flag=False), problem_smarts 실제 매치 확인. 페니토인의 인산에스터 프로드러그로, 체내 인산가수분해효소에 의한 에스터 절단 자체가 설계된 방출 메커니즘 - 이 경우 인산에스터 절단이 규칙이 우려하는 신경독성 반응성이 아니라 오히려 활성화 경로이므로, 프로드러그 맥락에서는 본 규칙의 무분별한 적용이 부적절할 수 있음을 실제 조회로 검증함.
- [부정_참고사례_검증필요] phosphor 규칙의 유일한 candidate(cleave_bond)는 에스터형 산소가 있는 유기인산 트리에스터(포스페이트)만 처리 가능함. valid set에서 phosphor로 진단된 33건 중 24건(73%)은 에스터 산소가 없는 트리아릴포스핀옥사이드 등 구조적으로 다른 화학종. ChEMBL 서브구조 검색(트리페닐포스핀옥사이드 코어, 62건 매치)에서 승인약물(max_phase=4) 0건 확인 — 이 골격은 약물 후보로 개발된 적이 거의 없는(유기합성 시약/부산물에 가까운) 계열로 보이며, 안전성 근거도 위험성 근거도 뚜렷하지 않아 새 치환 전략을 만들 화학적 정당성이 부족함. 이 서브클래스는 stuck이 정상적 결과로 남을 수 있음(범위 밖으로 명시, 억지 치환 candidate 추가는 지양).
✅ 스모크 테스트 통과

전체 통과 — 커밋해도 안전합니다.


In [24]:
!cd /content/laidd-2026 && git add -A && git commit -m "Document phosphor rule limitation: triarylphosphine oxides have no approved-drug precedent, defer new candidate (null-result)" && git push

[main 09f35b1] Document phosphor rule limitation: triarylphosphine oxides have no approved-drug precedent, defer new candidate (null-result)
 1 file changed, 10 insertions(+)
Enumerating objects: 9, done.
Counting objects: 100% (9/9), done.
Delta compression using up to 2 threads
Compressing objects: 100% (5/5), done.
Writing objects: 100% (5/5), 1.07 KiB | 1.07 MiB/s, done.
Total 5 (delta 3), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (3/3), completed with 3 local objects.
To https://github.com/Dec32th/laidd-2026.git
   c141d87..09f35b1  main -> main


In [33]:
importlib.reload(src.tools.agent)
importlib.reload(src.tools.molecule_editor)
from src.tools.molecule_editor import batch_iterative_fix_loop, clear_failure_memory
import inspect
print(inspect.signature(batch_iterative_fix_loop))

content = open('src/tools/molecule_editor.py').read()
print("시그니처에 use_debate 있는지:", 'use_debate=False,' in content)
print("_process_one 안에 use_debate 전달하는지:", content.count('use_debate=use_debate'))

(smiles_list, max_iterations=10, candidate_idx=0, llm_client=None, llm_model=None, llm_client_type='gemini', max_workers=5, progress=True)
시그니처에 use_debate 있는지: True
_process_one 안에 use_debate 전달하는지: 1


In [36]:
importlib.reload(src.tools.agent)
importlib.reload(src.tools.molecule_editor)
from src.tools.molecule_editor import batch_iterative_fix_loop, clear_failure_memory

import inspect
print(inspect.signature(batch_iterative_fix_loop))

(smiles_list, max_iterations=10, candidate_idx=0, llm_client=None, llm_model=None, llm_client_type='gemini', max_workers=5, progress=True)


In [34]:
importlib.reload(src.tools.agent)
importlib.reload(src.tools.molecule_editor)
from src.tools.molecule_editor import batch_iterative_fix_loop, clear_failure_memory
import inspect
print(inspect.signature(batch_iterative_fix_loop))

(smiles_list, max_iterations=10, candidate_idx=0, llm_client=None, llm_model=None, llm_client_type='gemini', max_workers=5, progress=True)


In [37]:
import src.tools.molecule_editor as me
print("로드된 파일 경로:", me.__file__)

with open(me.__file__) as f:
    content = f.read()
print("이 경로의 파일에 use_debate 있는지:", 'use_debate=False,' in content)

import subprocess
print(subprocess.run(['pwd'], capture_output=True, text=True).stdout)
print(subprocess.run(['find', '/content', '-maxdepth', '3', '-iname', 'laidd-2026', '-type', 'd'],
                      capture_output=True, text=True).stdout)

로드된 파일 경로: /content/laidd-2026/src/tools/molecule_editor.py
이 경로의 파일에 use_debate 있는지: True
/content/laidd-2026

/content/laidd-2026



In [38]:
importlib.reload(src.tools.agent)
importlib.reload(src.tools.molecule_editor)

from src.tools.molecule_editor import batch_iterative_fix_loop, iterative_fix_loop, clear_failure_memory

import inspect
print(inspect.signature(batch_iterative_fix_loop))

(smiles_list, max_iterations=10, candidate_idx=0, llm_client=None, llm_model=None, llm_client_type='gemini', max_workers=5, progress=True)


In [39]:
content = open('src/tools/molecule_editor.py').read()
count = content.count('def batch_iterative_fix_loop')
print(f"batch_iterative_fix_loop 정의 개수: {count} (1이어야 정상)")

if count > 1:
    import re
    for m in re.finditer(r'def batch_iterative_fix_loop\([^)]*\)', content):
        print(m.group())

batch_iterative_fix_loop 정의 개수: 2 (1이어야 정상)
def batch_iterative_fix_loop(smiles_list, max_iterations=10, candidate_idx=0,
                               llm_client=None, llm_model=None, llm_client_type="gemini",
                               max_workers=5, progress=True, use_debate=False,
                               debate_max_rounds=2)
def batch_iterative_fix_loop(smiles_list, max_iterations=10, candidate_idx=0,
                               llm_client=None, llm_model=None, llm_client_type="gemini",
                               max_workers=5, progress=True)


In [40]:
import re

path = 'src/tools/molecule_editor.py'
with open(path) as f:
    content = f.read()

# 파일 최상위 레벨의 'def ' 시작 위치를 전부 찾아서, 각 함수의 시작~끝 구간을 자름
def_starts = [m.start() for m in re.finditer(r'^def \w+', content, re.MULTILINE)]
def_starts.append(len(content))

blocks = []
for i in range(len(def_starts) - 1):
    block = content[def_starts[i]:def_starts[i+1]]
    if block.startswith('def batch_iterative_fix_loop'):
        blocks.append((def_starts[i], def_starts[i+1], block))

print(f"batch_iterative_fix_loop 블록 수: {len(blocks)}")
for start, end, block in blocks:
    has_use_debate = 'use_debate' in block
    print(f" - 위치 {start}~{end}, use_debate 포함: {has_use_debate}, 길이: {len(block)}자")

batch_iterative_fix_loop 블록 수: 2
 - 위치 15547~16882, use_debate 포함: True, 길이: 1335자
 - 위치 16882~18073, use_debate 포함: False, 길이: 1191자


In [41]:
old_blocks = [b for b in blocks if 'use_debate' not in b[2]]
assert len(old_blocks) == 1, f"제거 대상이 정확히 1개여야 하는데 {len(old_blocks)}개입니다"

start, end, block = old_blocks[0]
content_current = open('src/tools/molecule_editor.py').read()
new_content = content_current[:start] + content_current[end:]

with open('src/tools/molecule_editor.py', 'w') as f:
    f.write(new_content)

final_content = open('src/tools/molecule_editor.py').read()
print("정의 개수:", final_content.count('def batch_iterative_fix_loop'), "(1이어야 정상)")

import ast
ast.parse(final_content)
print("✅ 문법 정상")

정의 개수: 1 (1이어야 정상)
✅ 문법 정상


In [42]:
importlib.reload(src.tools.agent)
importlib.reload(src.tools.molecule_editor)
from src.tools.molecule_editor import batch_iterative_fix_loop, iterative_fix_loop, clear_failure_memory

import inspect
print(inspect.signature(batch_iterative_fix_loop))

(smiles_list, max_iterations=10, candidate_idx=0, llm_client=None, llm_model=None, llm_client_type='gemini', max_workers=5, progress=True, use_debate=False, debate_max_rounds=2)


In [43]:
!cd /content/laidd-2026 && git add -A && git commit -m "Pass use_debate/debate_max_rounds through batch_iterative_fix_loop" && git push

[main 9c89b1d] Pass use_debate/debate_max_rounds through batch_iterative_fix_loop
 1 file changed, 3 insertions(+), 27 deletions(-)
Enumerating objects: 9, done.
Counting objects: 100% (9/9), done.
Delta compression using up to 2 threads
Compressing objects: 100% (5/5), done.
Writing objects: 100% (5/5), 534 bytes | 534.00 KiB/s, done.
Total 5 (delta 3), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (3/3), completed with 3 local objects.
To https://github.com/Dec32th/laidd-2026.git
   09f35b1..9c89b1d  main -> main


In [44]:
import random, time
from collections import Counter

random.seed(7)
sample_200 = random.sample(list(data['smiles_valid']), 200)

from src.tools.agent import _llm_error_log
_llm_error_log.clear()
clear_failure_memory()

t0 = time.time()
results_debate = batch_iterative_fix_loop(
    sample_200, max_iterations=10, candidate_idx=0,
    llm_client=client_qwen, llm_model=QWEN_MODEL, llm_client_type="openai_compatible",
    max_workers=8, progress=False,
    use_debate=True, debate_max_rounds=2,
)
elapsed = time.time() - t0

status_counter = Counter(r['status'] for _, r in results_debate)
print(f"200개, use_debate=True, max_workers=8: {elapsed:.1f}초 ({elapsed/60:.1f}분)")
print(f"에러: {len(_llm_error_log)}건")
for status, count in status_counter.most_common():
    print(f"{status}: {count}")

debated_count = sum(1 for _, r in results_debate for h in r['history'] if h.get('debate_rounds'))
print(f"\n토의(debate)가 실제로 발동된 스텝 수: {debated_count}")

[06:41:41] Incomplete atom labelling, cannot make bond
[06:41:50] Incomplete atom labelling, cannot make bond


200개, use_debate=True, max_workers=8: 349.5초 (5.8분)
에러: 0건
success: 131
stuck: 43
no_known_fix: 26

토의(debate)가 실제로 발동된 스텝 수: 5


In [45]:
from src.tools.agent import _llm_error_log
_llm_error_log.clear()
clear_failure_memory()

t0 = time.time()
results_no_debate = batch_iterative_fix_loop(
    sample_200, max_iterations=10, candidate_idx=0,
    llm_client=client_qwen, llm_model=QWEN_MODEL, llm_client_type="openai_compatible",
    max_workers=8, progress=False,
    use_debate=False,
)
elapsed = time.time() - t0

status_counter_nd = Counter(r['status'] for _, r in results_no_debate)
print(f"200개, use_debate=False, max_workers=8: {elapsed:.1f}초 ({elapsed/60:.1f}분)")
print(f"에러: {len(_llm_error_log)}건")
for status, count in status_counter_nd.most_common():
    print(f"{status}: {count}")

# 결과가 달라진 분자 비교
results_debate_dict = dict(results_debate)
results_no_debate_dict = dict(results_no_debate)

diffs = [(smi, results_no_debate_dict[smi]['status'], results_debate_dict[smi]['status'])
         for smi in sample_200 if results_no_debate_dict[smi]['status'] != results_debate_dict[smi]['status']]
print(f"\n토의 유무로 결과가 달라진 분자: {len(diffs)}건")
for smi, nd, d in diffs:
    print(f"[{nd} -> {d}] {smi}")

[06:47:52] Incomplete atom labelling, cannot make bond


200개, use_debate=False, max_workers=8: 46.1초 (0.8분)
에러: 0건
success: 135
stuck: 38
no_known_fix: 27

토의 유무로 결과가 달라진 분자: 5건
[success -> stuck] O=CC=C(c1ccccc1)c1ccccc1
[success -> stuck] C=C(CC(=O)O)C(=O)O
[success -> stuck] Nc1cc(Cl)c(NC2=NCCN2)c(Cl)c1
[success -> stuck] CC(=O)/C=C/C=C1/C2CCC(C2)C1(C)C
[no_known_fix -> stuck] Nc1ccc(/N=N\c2ccccc2)c(N)c1


In [46]:
from src.tools.audit import generate_audit_report

for smi, nd, d in diffs:
    print(generate_audit_report(results_debate_dict[smi], original_smiles=smi))
    print("\n\n")

치환 감사추적 리포트
원본 분자: O=CC=C(c1ccccc1)c1ccccc1
최종 상태: stuck
최종 분자: NC(=O)C=C(c1ccccc1)c1ccccc1

--- 단계별 이력 ---

[스텝 0] O=CC=C(c1ccccc1)c1ccccc1
  진단된 문제: ['aldehyde', 'Michael_acceptor_1']

[스텝 1] NC(=O)C=C(c1ccccc1)c1ccccc1
  진단된 문제: ['Michael_acceptor_1']
  고친 규칙: aldehyde (판단 근거: 알데하이드는 마이클 수용체 시스템의 일부이므로 공액계를 먼저 제거하거나 변형해야 알데하이드의 반응성 및 독성이 근본적으로 해결됩니다.)
  적용된 치환: amide
  candidate 선택 근거: 참고사항이 없는 상황에서 알데히드의 친전자성 독성을 해결하기 위해 형태 유사성을 유지하면서 반응성을 제거하는 amide가 alcohol보다 약물 설계 관점에서 더 적절한 대체기이다. (candidate_idx=0, 완전 해소)

--- stuck 사유 ---
이 단계에서 known 규칙들의 모든 candidate를 순서대로 시도했으나 (['Michael_acceptor_1[idx=0](토의 결과 반려)']) 모두 실행에 실패했습니다(memory-skip 표시는 이전에 실패했던 것으로 확인되어 재시도 없이 건너뛴 항목). 흔한 원인: 유기금속/무기염 등 특수 화학종, 고리 구조와의 예상치 못한 충돌, 또는 원자가 계산 오류입니다.



치환 감사추적 리포트
원본 분자: C=C(CC(=O)O)C(=O)O
최종 상태: stuck
최종 분자: C=C(CC(=O)O)C(=O)O

--- 단계별 이력 ---

[스텝 0] C=C(CC(=O)O)C(=O)O
  진단된 문제: ['Michael_acceptor_1']

--- stuck 사유 ---
이 단계에서 known 규칙들의 모든 candidate를 순서대로 시도했으나 (['Michael_acceptor_1[idx=0](토의 결과 

In [47]:
from src.tools.agent import ask_llm_debate_fix, should_debate
from src.tools.molecule_editor import propose_fix

cases = [
    ("O=CC=C(c1ccccc1)c1ccccc1", "Michael_acceptor_1"),
    ("C=C(CC(=O)O)C(=O)O", "Michael_acceptor_1"),
    ("Nc1cc(Cl)c(NC2=NCCN2)c(Cl)c1", "aniline"),
    ("CC(=O)/C=C/C=C1/C2CCC(C2)C1(C)C", "Michael_acceptor_1"),
    ("Nc1ccc(NNc2ccccc2)c(N)c1", "aniline"),  # azo 규칙 처리 후 상태
]

for smi, rule in cases:
    info = get_replacement_candidates(rule)
    candidate = info['candidates'][0]
    trial = propose_fix(smi, rule, 0)
    if trial is None:
        print(f"[{rule}] {smi} -> propose_fix 자체가 실패(candidate 없음)")
        continue
    print(f"\n{'='*50}\n분자: {smi}\n규칙: {rule}\ncandidate: {candidate['name']}")
    result = ask_llm_debate_fix(
        client_qwen, QWEN_MODEL, smi, trial['new_smiles'], rule,
        candidate['name'], candidate.get('rationale', ''),
        client_type="openai_compatible", max_rounds=2,
    )
    print(f"최종 판정: {result['final_verdict']}")
    for r in result['rounds']:
        print(f"  [{r['role']} R{r['round']}] {r['text']}")


분자: O=CC=C(c1ccccc1)c1ccccc1
규칙: Michael_acceptor_1
candidate: saturated (C-C single bond)
최종 판정: rejected
  [critic R1] {'verdict': 'rejected', 'reason': "제안된 치환은 Michael acceptor 경고 해결이라는 명분은 타당하나, EGFR 표적에 대한 결합력 손실과 약리 기전 불일치라는 치명적 결함이 있습니다. 1) 도킹 스코어가 -6.14에서 -5.95 kcal/mol로 악화되었으며, 이는 공유결합 억제제(covalent inhibitor) 설계 시 비공유 상호작용의 최적화가 필수적임에도 불구하고 포화(saturation)가 결합 친화도를 저해했음을 의미합니다. 2) 제안자가 인용한 에타크린산(Ethacrynic acid) 사례는 오히려 역설적입니다. 에타크린산은 알파,베타-불포화 카르보닐 구조 그 자체가 시스테인 잔기와 반응하여 효능을 발현하는 warhead입니다. 이를 환원하여 제거한다는 것은, 만약 해당 분자가 EGFR에 대해 공유결합 기전으로 작용하려 한다면 '약효 자체를 거세(castration)'하는 행위가 됩니다. 3) PAINS 점수는 1.0으로 개선되었으나, QED(0.65)와 Lipinski(0.5)가 평범한 수준에 머물러 있어 단순한 구조 포화가 약물성(drug-likeness)을 담보하지 못합니다. 결론적으로, 비공유 결합력이 감소했고 warhead 제거가 의도된 것인지 혹은 오류인지 모호하므로, 현재 데이터만으로는 승인할 수 없습니다."}
  [proposer R1] {'stance': 'withdraw', 'argument': '동료 검토자의 지적이 타당합니다. 본 치환은 알파,베타-불포화 카르보닐이라는 공유결합 억제제의 핵심 warhead를 제거하여 PAINS 경고는 해결했으나, 이는 EGFR 표적에 대한 약리 기전 자체를 무효화하는 결과를 초래했습니다. 도킹 스코어 악화는 비공유 결합력 저하를 입증하며,

In [48]:
! cat src/tools/docking.py

"""도킹 자동화 파이프라인.
표적별 HETATM 처리(보조인자 유지 여부)는 반드시 사람이 먼저 확인해야
하므로, 새 표적을 추가할 때는 DOCKING_TARGETS에 keep_hetatm_codes를
명시적으로 등록하는 과정을 거친다(완전 자동화 금지 지점)."""

import os
import json
import re
import subprocess
from rdkit import Chem
from rdkit.Chem import AllChem

VINA_BIN = os.path.join(os.getcwd(), "vina_bin")

DOCKING_TARGETS = {
    "catechol": {
        "target_name": "COMT", "pdb_id": "1VID",
        "ligand_code": "DNC", "keep_hetatm_codes": ["MG", "SAM"],
        "box_size": [20, 20, 20],
    },
    "Michael_acceptor_1": {
        "target_name": "EGFR", "pdb_id": "6JX4",
        "ligand_code": "YY3", "keep_hetatm_codes": [],
        "box_size": [20, 20, 20],
    },
    "hydroquinone": {
        "target_name": "NQO1", "pdb_id": "1DXO",
        "ligand_code": "DQN", "keep_hetatm_codes": ["FAD"],
        "box_size": [20, 20, 20],
    },
    "quinone_A(370)": {
        "target_name": "NQO1", "pdb_id": "1DXO",
        "ligand_code": "DQN", "keep_hetatm_codes": ["FAD"],
        "box_size": [

In [6]:
%%writefile src/tools/docking.py
"""도킹 자동화 파이프라인.
표적별 HETATM 처리(보조인자 유지 여부)는 반드시 사람이 먼저 확인해야
하므로, 새 표적을 추가할 때는 DOCKING_TARGETS에 keep_hetatm_codes를
명시적으로 등록하는 과정을 거친다(완전 자동화 금지 지점)."""

import os
import json
import re
import subprocess
from rdkit import Chem
from rdkit.Chem import AllChem

VINA_BIN = os.path.join(os.getcwd(), "vina_bin")

DOCKING_TARGETS = {
    "catechol": {
        "target_name": "COMT", "pdb_id": "1VID",
        "ligand_code": "DNC", "keep_hetatm_codes": ["MG", "SAM"],
        "box_size": [20, 20, 20],
        "caveat": None,  # 매치된 분자가 실제로 카테콜 골격을 가지므로 표적 특이성 근거 있음
    },
    "Michael_acceptor_1": {
        "target_name": "EGFR", "pdb_id": "6JX4",
        "ligand_code": "YY3", "keep_hetatm_codes": [],
        "box_size": [20, 20, 20],
        "caveat": ("EGFR은 Michael_acceptor_1 규칙 자체와 특이적 연관이 없는 벤치마크 표적입니다. "
                   "이 특정 분자가 실제로 EGFR을 겨냥한다는 근거는 없으므로, 이 도킹 결과를 "
                   "'이 분자는 EGFR 공유결합 억제제다'라는 근거로 쓰지 마세요. 참고로만 활용하고, "
                   "warhead 제거가 약효를 없앤다는 결론은 이 분자가 실제로 공유결합 표적을 "
                   "가진다는 별도 증거가 있을 때만 내리세요."),
    },
    "hydroquinone": {
        "target_name": "NQO1", "pdb_id": "1DXO",
        "ligand_code": "DQN", "keep_hetatm_codes": ["FAD"],
        "box_size": [20, 20, 20],
        "caveat": None,
    },
    "quinone_A(370)": {
        "target_name": "NQO1", "pdb_id": "1DXO",
        "ligand_code": "DQN", "keep_hetatm_codes": ["FAD"],
        "box_size": [20, 20, 20],
        "caveat": None,
    },
}

_CACHE_PATH = "outputs/docking_cache.json"


def _load_cache():
    if os.path.exists(_CACHE_PATH):
        with open(_CACHE_PATH) as f:
            return json.load(f)
    return {}


def _save_cache(cache):
    os.makedirs("outputs", exist_ok=True)
    with open(_CACHE_PATH, "w") as f:
        json.dump(cache, f, ensure_ascii=False, indent=2)


def prepare_ligand_pdbqt(smiles, filename, seed=42):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    mol = Chem.AddHs(mol)
    if AllChem.EmbedMolecule(mol, randomSeed=seed) != 0:
        return None
    AllChem.MMFFOptimizeMolecule(mol)
    Chem.MolToPDBFile(mol, f"{filename}.pdb")
    subprocess.run(["obabel", f"{filename}.pdb", "-O", f"{filename}.pdbqt"], capture_output=True)
    out_path = f"{filename}.pdbqt"
    return out_path if os.path.exists(out_path) else None


def run_docking_cli(ligand_pdbqt, receptor_pdbqt, out_prefix, box_center, box_size, exhaustiveness=4):
    log_path = f"{out_prefix}_log.txt"
    cmd = [
        VINA_BIN, "--receptor", receptor_pdbqt, "--ligand", ligand_pdbqt,
        "--center_x", str(box_center[0]), "--center_y", str(box_center[1]), "--center_z", str(box_center[2]),
        "--size_x", str(box_size[0]), "--size_y", str(box_size[1]), "--size_z", str(box_size[2]),
        "--exhaustiveness", str(exhaustiveness), "--out", f"{out_prefix}_out.pdbqt",
    ]
    try:
        with open(log_path, "w") as log_f:
            subprocess.run(cmd, stdout=log_f, stderr=subprocess.STDOUT, timeout=180)
    except subprocess.TimeoutExpired:
        return None

    if not os.path.exists(log_path):
        return None
    with open(log_path) as f:
        log = f.read()
    match = re.search(r"^\s*1\s+(-?\d+\.\d+)", log, re.MULTILINE)
    return float(match.group(1)) if match else None


def prepare_receptor(pdb_id, keep_hetatm_codes):
    """수용체 준비. keep_hetatm_codes는 반드시 사람이 그 표적의 HETATM
    목록(prepare_receptor 호출 전 raw PDB를 먼저 열어 확인)을 보고
    직접 지정해야 하며, 빈 리스트라도 명시적으로 넘겨야 한다."""
    os.makedirs("targets", exist_ok=True)
    raw_path = f"targets/{pdb_id}.pdb"
    if not os.path.exists(raw_path):
        subprocess.run(["wget", "-q", f"https://files.rcsb.org/download/{pdb_id}.pdb", "-O", raw_path])

    with open(raw_path) as f:
        lines = f.readlines()

    keep_set = set(keep_hetatm_codes)
    clean_lines = [l for l in lines if l.startswith(("ATOM", "TER", "END"))
                   or (l.startswith("HETATM") and l[17:20].strip() in keep_set)]
    clean_path = f"targets/{pdb_id}_clean.pdb"
    with open(clean_path, "w") as f:
        f.writelines(clean_lines)

    receptor_pdbqt = f"targets/{pdb_id}_receptor.pdbqt"
    if not os.path.exists(receptor_pdbqt):
        subprocess.run(["obabel", clean_path, "-O", receptor_pdbqt, "-xr"], capture_output=True)
    return receptor_pdbqt, lines


def inspect_hetatm(pdb_id):
    """새 표적을 DOCKING_TARGETS에 등록하기 전, HETATM 목록을 먼저
    확인하는 용도. 자동 결정 없이 사람이 보고 keep_hetatm_codes를
    정하도록 정보만 제공한다."""
    os.makedirs("targets", exist_ok=True)
    raw_path = f"targets/{pdb_id}.pdb"
    if not os.path.exists(raw_path):
        subprocess.run(["wget", "-q", f"https://files.rcsb.org/download/{pdb_id}.pdb", "-O", raw_path])
    with open(raw_path) as f:
        lines = f.readlines()
    hetero = set(l[17:20].strip() for l in lines if l.startswith("HETATM"))
    return hetero


def auto_dock_precedent(rule_name, original_smiles, fixed_smiles, use_cache=True):
    """rule_name으로 DOCKING_TARGETS에서 표적 정보를 찾아 도킹 실행.
    pdb_id/ligand_code가 None이면 아직 사람 확인이 안 된 표적이므로
    명확히 에러를 반환한다(자동으로 대충 진행하지 않음)."""
    if rule_name not in DOCKING_TARGETS:
        return {"error": f"'{rule_name}'은 DOCKING_TARGETS에 등록되지 않음"}

    target_info = DOCKING_TARGETS[rule_name]
    if target_info["pdb_id"] is None or target_info["ligand_code"] is None:
        return {"error": f"'{rule_name}' 표적의 pdb_id/ligand_code가 아직 "
                          f"확인 안 됨. inspect_hetatm()으로 먼저 확인 후 "
                          f"DOCKING_TARGETS를 채워주세요."}

    cache = _load_cache() if use_cache else {}
    cache_key = f"{rule_name}|{original_smiles}|{fixed_smiles}"
    if use_cache and cache_key in cache:
        return cache[cache_key]

    if not os.path.exists(VINA_BIN):
        return {"error": f"vina_bin이 {VINA_BIN}에 없음. 다운로드 먼저 진행하세요."}

    receptor_pdbqt, lines = prepare_receptor(target_info["pdb_id"], target_info["keep_hetatm_codes"])

    ligand_lines = [l for l in lines if l.startswith("HETATM") and l[17:20].strip() == target_info["ligand_code"]]
    if not ligand_lines:
        return {"error": f"리간드 코드 {target_info['ligand_code']}를 PDB에서 찾을 수 없음"}
    coords = [(float(l[30:38]), float(l[38:46]), float(l[46:54])) for l in ligand_lines]
    box_center = [sum(c[i] for c in coords) / len(coords) for i in range(3)]

    scores = {}
    for label, smi in [("original", original_smiles), ("fixed", fixed_smiles)]:
        lig_pdbqt = prepare_ligand_pdbqt(smi, f"targets/{rule_name}_{label}")
        if lig_pdbqt is None:
            return {"error": f"{label} 리간드 준비 실패"}
        scores[label] = run_docking_cli(lig_pdbqt, receptor_pdbqt, f"targets/{rule_name}_{label}",
                                          box_center, target_info["box_size"])

    result = {
        "target": target_info["target_name"], "pdb_id": target_info["pdb_id"], "rule": rule_name,
        "score_original": scores["original"], "score_fixed": scores["fixed"],
        "delta": (scores["fixed"] - scores["original"])
                 if scores["original"] is not None and scores["fixed"] is not None else None,
        "caveat": target_info.get("caveat"),
    }

    if use_cache:
        cache[cache_key] = result
        _save_cache(cache)

    return result


Overwriting src/tools/docking.py


In [7]:
!cat src/tools/agent.py


import json
import time
from src.tools.replacement_library import get_replacement_candidates

_llm_error_log = []
_llm_consecutive_failures = 0
_LLM_FAILURE_LIMIT = 3
_debate_call_budget = {"remaining": 100}

def set_debate_budget(n):
    """토의(debate)에 쓸 수 있는 총 LLM 호출 수 상한을 재설정."""
    _debate_call_budget["remaining"] = n

_injected_sascorer = {"module": None}
_injected_tox_predictor = {"fn": None}

def set_sascorer_module(module):
    """세션마다 다운로드/import한 sascorer 모듈을 등록."""
    _injected_sascorer["module"] = module

def set_tox_predictor(fn):
    """(original_smiles, fixed_smiles, rule_name) -> tox_delta(float) 또는 None
    을 반환하는 콜백을 등록. 노트북마다 다르게 학습한 baseline 모델을
    감싸서 넘기면 됨."""
    _injected_tox_predictor["fn"] = fn

def _call_llm(client, model_name, prompt, client_type="gemini"):
    """client_type에 따라 Gemini SDK 또는 OpenAI 호환 SDK로 호출하고,
    응답 텍스트만 통일된 형태로 반환."""
    if client_type == "gemini":
        response = client.models.generate_content(model=model_name, contents=prompt

In [10]:
%%writefile src/tools/agent.py
import json
import time
from src.tools.replacement_library import get_replacement_candidates

_llm_error_log = []
_llm_consecutive_failures = 0
_LLM_FAILURE_LIMIT = 3
_debate_call_budget = {"remaining": 100}

def set_debate_budget(n):
    """토의(debate)에 쓸 수 있는 총 LLM 호출 수 상한을 재설정."""
    _debate_call_budget["remaining"] = n

_injected_sascorer = {"module": None}
_injected_tox_predictor = {"fn": None}

def set_sascorer_module(module):
    """세션마다 다운로드/import한 sascorer 모듈을 등록."""
    _injected_sascorer["module"] = module

def set_tox_predictor(fn):
    """(original_smiles, fixed_smiles, rule_name) -> tox_delta(float) 또는 None
    을 반환하는 콜백을 등록. 노트북마다 다르게 학습한 baseline 모델을
    감싸서 넘기면 됨."""
    _injected_tox_predictor["fn"] = fn

def _call_llm(client, model_name, prompt, client_type="gemini"):
    """client_type에 따라 Gemini SDK 또는 OpenAI 호환 SDK로 호출하고,
    응답 텍스트만 통일된 형태로 반환."""
    if client_type == "gemini":
        response = client.models.generate_content(model=model_name, contents=prompt)
        return response.text
    elif client_type == "openai_compatible":
        for attempt in range(2):  # rate limit 시 1회만 재시도
            try:
                response = client.chat.completions.create(
                    model=model_name,
                    messages=[{"role": "user", "content": prompt}],
                    max_tokens=500,
                    timeout=30,
                    extra_body={"enable_thinking": False},
                )
                return response.choices[0].message.content
            except Exception as e:
                _llm_error_log.append(repr(e))
                if 'RateLimitError' in type(e).__name__ and attempt == 0:
                    time.sleep(3)
                    continue
                return f"ERROR: LLM 호출 실패/타임아웃 - {e}"
    else:
        raise ValueError(f"알 수 없는 client_type: {client_type}")


def _parse_json_response(text, fallback):
    text = text.strip()
    if text.startswith('```'):
        text = text.split('```')[1]
        if text.startswith('json'):
            text = text[4:]
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return fallback

def _try_get_docking_evidence(rule_name, smiles_before, smiles_after):
    """도킹 표적이 등록된 규칙이면 자동으로 도킹 실행, 아니면 None.
    도킹 실패/미등록/예외는 전부 조용히 None으로 처리(critic 프롬프트에서
    도킹 근거 없이 진행하는 것으로 자연스럽게 폴백)."""
    try:
        from src.tools.docking import auto_dock_precedent, DOCKING_TARGETS
        if rule_name not in DOCKING_TARGETS:
            return None
        result = auto_dock_precedent(rule_name, smiles_before, smiles_after)
        if result.get('error') or result.get('delta') is None:
            return None
        return result
    except Exception:
        return None

def _try_compute_score(rule_name, smiles_before, smiles_after, docking_evidence=None):
    """등록된 sascorer/tox_predictor/도킹 결과를 모아 종합 점수 계산.
    일부만 등록돼 있어도 compute_multi_objective_score가 나머지로
    자동 정규화하므로 실패하지 않음. 계산 자체가 실패하면 None."""
    try:
        from src.tools.scoring import compute_multi_objective_score

        tox_delta = None
        if _injected_tox_predictor["fn"] is not None:
            try:
                tox_delta = _injected_tox_predictor["fn"](smiles_before, smiles_after, rule_name)
            except Exception:
                tox_delta = None

        precedent_docking_delta = docking_evidence["delta"] if docking_evidence else None

        return compute_multi_objective_score(
            smiles_before, smiles_after, rule_name,
            tox_delta=tox_delta,
            sascorer_module=_injected_sascorer["module"],
            precedent_docking_delta=precedent_docking_delta,
        )
    except Exception:
        return None

def _try_get_activity_risk(rule_name, smiles_before, smiles_after):
    """활성 보존 위험도 평가. sascorer가 등록 안 돼 있으면 조용히 None
    (activity_metrics가 sascorer_module을 필수로 요구하므로)."""
    try:
        if _injected_sascorer["module"] is None:
            return None
        from src.tools.activity_metrics import (compute_activity_preservation_metrics,
                                                   classify_activity_risk_v3)
        metrics = compute_activity_preservation_metrics(
            smiles_before, smiles_after, _injected_sascorer["module"]
        )
        if metrics is None:
            return None
        return classify_activity_risk_v3(metrics)
    except Exception:
        return None

def ask_llm_which_problem_to_fix(client, model_name, smiles, problems, client_type="gemini"):
    """여러 toxicophore 중 어떤 것부터 고칠지 LLM에게 판단을 요청."""
    known = [p for p in problems if get_replacement_candidates(p['rule_name']) is not None]

    if not known:
        return None
    if len(known) == 1:
        return {"rule_name": known[0]['rule_name'], "reason": "유일한 치환 가능 후보"}

    prompt = f"""당신은 신약개발 화학자입니다. 다음 분자에서 여러 구조적 문제(toxicophore)가 발견되었습니다.

분자 SMILES: {smiles}

발견된 문제 중, 우리가 실제로 치환 가능한 것들:
{json.dumps(known, ensure_ascii=False, indent=2)}

이 중 어떤 문제를 먼저 해결하는 것이 화학적으로 더 타당한지 판단하고,
반드시 아래 JSON 형식으로만 답하세요. 다른 설명 없이 JSON만 출력하세요.

{{"rule_name": "선택한 문제의 rule_name", "reason": "선택 이유 한 문장"}}
"""

    text = _call_llm(client, model_name, prompt, client_type)
    fallback = {"rule_name": known[0]['rule_name'], "reason": "JSON 파싱 실패, 기본값(첫 번째 후보) 사용"}
    return _parse_json_response(text, fallback)


def ask_llm_which_candidate_to_use(client, model_name, smiles, rule_name, client_type="gemini"):
    """한 문제(rule_name)에 대한 여러 치환 후보 중 어떤 걸 쓸지 LLM에게 판단 요청.

    candidate의 rationale 중 하나라도 '[참고]'로 시작하는 문구가 있으면,
    이는 실제 승인약물 사례에서 이 골격이 안전하게 쓰인 경우가 있다는 뜻이므로,
    candidate가 1개뿐이더라도(원래는 LLM 호출을 건너뛰던 경우) 반드시 LLM에게
    판단을 맡긴다. 이 경우 LLM은 candidate_idx로 -1을 반환하여 "치환을
    보류하고 사람(연구자) 검토가 필요하다"고 명시적으로 표시할 수 있다.
    """
    info = get_replacement_candidates(rule_name)
    if info is None:
        return None

    candidates = info['candidates']
    has_caution = any('[참고]' in c.get('rationale', '') for c in candidates)

    if len(candidates) == 1 and not has_caution:
        return {"candidate_idx": 0, "reason": "유일한 후보"}

    candidate_info = [
        {"idx": i, "name": c['name'], "rationale": c['rationale']}
        for i, c in enumerate(candidates)
    ]

    prompt = f"""당신은 신약개발 화학자입니다. 다음 분자에서 '{rule_name}' 문제를
해결하기 위한 치환 후보가 있습니다.

분자 SMILES: {smiles}

치환 후보들:
{json.dumps(candidate_info, ensure_ascii=False, indent=2)}

각 후보의 rationale에 "[참고]"로 시작하는 문구가 있다면, 이는 "이 골격이
실제 승인 약물에서 반응성이 아닌 안정적 형태로 널리 쓰인 사례가 있으니,
경고를 절대적 기준이 아닌 참고 신호로 해석하라"는 뜻입니다. 이 경우 먼저
"이 분자가 그 참고사항이 가리키는 안전한 사용 사례와 실제로 유사한지"를
판단하세요.
- 유사하다고 판단되면서, 후보가 여러 개라면 변화 폭이 더 작은 후보를 선택하세요.
- 유사하다고 판단되고, 치환 자체가 불필요하다고 볼 만큼 뚜렷하다면,
  candidate_idx를 -1로 답해 "치환 보류, 사람 검토 필요"를 표시하세요.
- 참고사항이 없거나 이 분자가 그 사례와 유사하지 않다면, 평소대로 가장
  적절한 후보를 선택하세요.

반드시 아래 JSON 형식으로만 답하세요. 다른 설명 없이 JSON만 출력하세요.

{{"candidate_idx": 선택한 후보의 idx(정수, 또는 보류 시 -1), "reason": "판단 이유 한 문장"}}
"""

    text = _call_llm(client, model_name, prompt, client_type)
    fallback = {"candidate_idx": 0, "reason": "JSON 파싱 실패, 기본값(첫 번째 후보) 사용"}
    result = _parse_json_response(text, fallback)

    idx = result.get('candidate_idx')
    if not isinstance(idx, int) or not (-1 <= idx < len(candidates)):
        return {"candidate_idx": 0, "reason": "LLM 응답 idx 범위 오류, 기본값 사용"}
    return result


def ask_llm_debate_fix(client, model_name, smiles_before, smiles_after, rule_name,
                        candidate_name, candidate_rationale, client_type="gemini",
                        max_rounds=2):
    """제안자(원래 candidate를 고른 논리)와 검토자(critic)가 여러 라운드
    대화하며 합의에 도달하려 시도. 매 라운드 critic이 판단하고, 반려하면
    proposer가 반박, critic이 재판단. max_rounds 안에 합의(양쪽 다 승인,
    또는 critic이 최종 반려로 확정) 안 되면 "escalate"로 사람 검토行.

    반환: {"final_verdict": "approved"|"rejected"|"escalate",
           "rounds": [{"role": "critic"|"proposer", "text": str}, ...],
           "consensus_reached": bool}
    """
    if _debate_call_budget["remaining"] <= 0:
        return {"final_verdict": "approved", "rounds": [], "consensus_reached": True,
                "budget_exhausted": True}
    _debate_call_budget["remaining"] -= 1
    rounds_log = []
    proposer_argument = candidate_rationale

    for round_num in range(1, max_rounds + 1):
        docking_evidence = _try_get_docking_evidence(rule_name, smiles_before, smiles_after)
        score_result = _try_compute_score(rule_name, smiles_before, smiles_after, docking_evidence)
        activity_risk = _try_get_activity_risk(rule_name, smiles_before, smiles_after)
        critic_prompt = f"""당신은 신약개발 화학 검토자(critic)입니다. 동료 화학자가 아래
치환을 제안했습니다.

원본 분자: {smiles_before}
치환 후 분자: {smiles_after}
해결하려던 문제: {rule_name}
제안된 치환: {candidate_name}
제안자의 근거: {proposer_argument}
{f"실측 도킹 결합력 변화: {docking_evidence['target']} 표적, {docking_evidence['score_original']:.2f} → {docking_evidence['score_fixed']:.2f} kcal/mol (delta {docking_evidence['delta']:+.2f}). 이 정량 데이터를 판단에 반영하세요.{' [주의: ' + docking_evidence['caveat'] + ']' if docking_evidence and docking_evidence.get('caveat') else ''}" if docking_evidence else ""}
{f"종합 점수: {score_result['composite_score']:.2f} (세부: {score_result['component_scores']}). 이것도 판단에 참고하세요." if score_result and score_result.get('composite_score') is not None else ""}
{f"활성 보존 위험도 평가: {activity_risk['verdict']} (세부: {'; '.join(activity_risk['details'])})" if activity_risk else ""}

이 치환에 동의하는지 비판적으로 검토하세요. 동의하지 않는다면 구체적으로
어떤 점이 문제인지 명시하세요(새로운 독성 구조 생성 가능성, 근거의
논리적 결함, precedent 오독 등).

반드시 아래 JSON 형식으로만 답하세요.
{{"verdict": "approved" 또는 "rejected", "reason": "판단 이유, 반려 시 구체적 반론 포함"}}
"""
        critic_text = _call_llm(client, model_name, critic_prompt, client_type)
        critic_result = _parse_json_response(
            critic_text, {"verdict": "approved", "reason": "JSON 파싱 실패, 기본 승인"}
        )
        rounds_log.append({"role": "critic", "round": round_num, "text": critic_result})

        if critic_result.get("verdict") == "approved":
            return {"final_verdict": "approved", "rounds": rounds_log, "consensus_reached": True}

        if round_num == max_rounds:
            break

        proposer_prompt = f"""당신은 방금 아래 치환을 제안한 화학자입니다.

원본 분자: {smiles_before}
치환 후 분자: {smiles_after}
당신의 원래 근거: {proposer_argument}

동료 검토자(critic)가 다음과 같이 반려했습니다: "{critic_result.get('reason', '')}"

이 반론에 대해 답하세요. 반론이 타당하면 인정하고 제안을 철회하세요.
반론이 부당하다면 왜 원래 치환이 여전히 타당한지 반박하세요.

반드시 아래 JSON 형식으로만 답하세요.
{{"stance": "withdraw" 또는 "defend", "argument": "반박 또는 철회 이유"}}
"""
        proposer_text = _call_llm(client, model_name, proposer_prompt, client_type)
        proposer_result = _parse_json_response(
            proposer_text, {"stance": "withdraw", "argument": "JSON 파싱 실패, 기본 철회"}
        )
        rounds_log.append({"role": "proposer", "round": round_num, "text": proposer_result})

        if proposer_result.get("stance") == "withdraw":
            return {"final_verdict": "rejected", "rounds": rounds_log, "consensus_reached": True}

        proposer_argument = proposer_result.get("argument", proposer_argument)

    return {"final_verdict": "escalate", "rounds": rounds_log, "consensus_reached": False}
def should_debate(candidate_rationale):
    """이 candidate가 토의(debate)를 거칠 필요가 있는지 판단.
    rationale에 '[참고]'가 있으면 실제 승인약물 사례와 겹칠 수 있다는
    뜻이므로, 단순 채택 대신 토의로 한 번 더 검토해야 함."""
    return '[참고]' in (candidate_rationale or '')


Overwriting src/tools/agent.py


In [11]:
import ast
with open('src/tools/docking.py') as f:
    ast.parse(f.read())
print("✅ docking.py 문법 정상")
with open('src/tools/agent.py') as f:
    content = f.read()
    ast.parse(content)
print("✅ agent.py 문법 정상")
print("✅ caveat 반영:", "caveat" in content)

importlib.reload(src.tools.docking)
importlib.reload(src.tools.agent)
from src.tools.docking import _load_cache, _save_cache
from src.tools.agent import ask_llm_debate_fix

# 캐시 정리 (옛날 caveat 없는 결과 제거)
cache = _load_cache()
removed = [k for k in cache if k.startswith("Michael_acceptor_1|")]
for k in removed:
    del cache[k]
_save_cache(cache)
print(f"제거된 캐시: {removed}")

# 재확인: 이타콘산 케이스로 다시 토의
from src.tools.molecule_editor import propose_fix
smi = "C=C(CC(=O)O)C(=O)O"
info = get_replacement_candidates('Michael_acceptor_1')
candidate = info['candidates'][0]
trial = propose_fix(smi, 'Michael_acceptor_1', 0)
result = ask_llm_debate_fix(
    client_qwen, QWEN_MODEL, smi, trial['new_smiles'], 'Michael_acceptor_1',
    candidate['name'], candidate.get('rationale', ''),
    client_type="openai_compatible", max_rounds=2,
)
print(f"최종 판정: {result['final_verdict']}")
for r in result['rounds']:
    print(f"  [{r['role']} R{r['round']}] {r['text']}")

✅ docking.py 문법 정상
✅ agent.py 문법 정상
✅ caveat 반영: True
제거된 캐시: ['Michael_acceptor_1|C=C(CC(=O)O)C(=O)O|CC(CC(=O)O)C(=O)O']
최종 판정: rejected
  [critic R1] {'verdict': 'rejected', 'reason': "제안된 치환은 Michael acceptor 경고를 해결하지만, 분자의 핵심 약력단(pharmacophore)인 α,β-불포화 카르보닐 구조를 완전히 제거하여 공유결합 억제제로서의 기능을 상실시킵니다. 제안자가 인용한 에타크린산(Ethacrynic acid) 사례는 오히려 해당 warhead가 필수적임을 증명하는 반례입니다: 에타크린산은 C=C 이중결합을 통해 시스테인 잔기와 Michael 부가반응을 일으켜 이뇨 효과를 발휘하므로, 이를 포화 결합으로 환원하면 약효 기전 자체가 소멸됩니다. 도킹 점수의 미미한 개선(-0.12 kcal/mol)은 비특이적 EGFR 벤치마크 결과일 뿐이며, 실제 공유결합 표적에 대한 친전자성 반응성을 대체할 수 있는 증거가 없습니다. 또한 PAINS 점수는 1.0으로 유지되어 구조적 위장만 이루어졌을 뿐 근본적인 리스크 프로파일링이 완료되지 않았음을 시사합니다. Warhead 제거는 '최적화'가 아닌 '활성 포기'에 해당하므로, 공유결합 억제가 필수라면 soft electrophile로의 변환 등 반응성은 유지하면서 선택성을 높이는 대안을 검토해야 합니다."}
  [proposer R1] {'stance': 'withdraw', 'argument': '동료 검토자의 지적이 타당합니다. 제안된 치환(C=C → CC)은 α,β-불포화 카르보닐이라는 공유결합 억제제의 핵심 warhead를 제거하여, Michael addition을 통한 표적 결합 메커니즘 자체를 소멸시킵니다. 에타크린산 사례 인용은 오히려 해당 구조의 필수성을 증명하는 반례였으며, PAINS 점수 유지와 미미한 도킹 점수 개선은 활성 포기에

In [12]:
remaining_cases = [
    ("O=CC=C(c1ccccc1)c1ccccc1", "Michael_acceptor_1"),
    ("CC(=O)/C=C/C=C1/C2CCC(C2)C1(C)C", "Michael_acceptor_1"),
]

for smi, rule in remaining_cases:
    info = get_replacement_candidates(rule)
    candidate = info['candidates'][0]
    trial = propose_fix(smi, rule, 0)
    result = ask_llm_debate_fix(
        client_qwen, QWEN_MODEL, smi, trial['new_smiles'], rule,
        candidate['name'], candidate.get('rationale', ''),
        client_type="openai_compatible", max_rounds=2,
    )
    print(f"\n{smi}\n최종 판정: {result['final_verdict']}")
    for r in result['rounds']:
        print(f"  [{r['role']} R{r['round']}] {r['text']['reason'] if 'reason' in r['text'] else r['text']}")


O=CC=C(c1ccccc1)c1ccccc1
최종 판정: rejected
  [critic R1] 제안된 치환은 Michael acceptor 경고를 해소하려는 의도는 타당하나, 분자의 핵심 약력단(pharmacophore)인 공액(conjugation) 시스템을 파괴하여 화합물의 정체성을 근본적으로 변경시켰습니다. 1) 구조적 문제: 원본의 α,β-불포화 카르보닐은 단백질과의 공유결합 warhead이자 분자 전체의 평면성과 전자 분포를 유지하는 핵심 구조입니다. 이를 포화 결합으로 환원하면 warhead 기능이 소실될 뿐만 아니라, 두 페닐 고리와 카르보닐 간의 π-공액이 단절되어 3차원 형태와 물리화학적 성질이 완전히 달라집니다. 2) 도킹 데이터의 오해석 위험: EGFR 도킹 점수 변화(+0.20 kcal/mol)는 미미하지만, 이는 해당 분자가 EGFR 억제제라는 검증 없이 얻어진 벤치마크 결과일 뿐입니다. 실제 표적이 공유결합 억제제를 필요로 하는 경우라면, 이 치환은 약효를 '유지'한 것이 아니라 '소거'한 것입니다. 3) 제안 근거의 논리적 결함: 에타크린산 등의 예외 조항은 '공유결합이 필수적인 경우 경고를 무시해도 된다'는 뜻이지, '공유결합 부위를 제거해도 된다'는 의미가 아닙니다. PAINS 점수가 1.0으로 개선되었으나, 이는 유효한 리간드 설계를 포기하고 안전성 지표만 확보한 것에 불과합니다. 따라서 본 치환은 독성 우려를 해결하기보다 후보 물질로서의 가치를 상실하게 만들므로 반려합니다.
  [proposer R1] {'stance': 'withdraw', 'argument': '동료 검토자의 지적을 전적으로 수용합니다. 원본 분자의 α,β-불포화 카르보닐은 단순한 독성 경고(PAINS)의 대상이 아니라, 해당 화합물이 공유결합 억제제로서 기능하기 위한 필수 약력단(warhead)이자 분자 전체의 공액 시스템을 유지하는 핵심 구조입니다. 이를 포화 결합으로 환원하는 것은 독성을 제거하는 것이 아니라 약효 기전 자체를 소거하는 행위로, 

In [13]:
!cd /content/laidd-2026 && git add -A && git commit -m "Add caveat field to docking targets lacking rule-specific relevance (Michael_acceptor_1/EGFR); critic no longer falsely asserts specific target mechanism" && git push

[main 6d239e6] Add caveat field to docking targets lacking rule-specific relevance (Michael_acceptor_1/EGFR); critic no longer falsely asserts specific target mechanism
 5 files changed, 58 insertions(+), 31 deletions(-)
Enumerating objects: 21, done.
Counting objects: 100% (21/21), done.
Delta compression using up to 2 threads
Compressing objects: 100% (11/11), done.
Writing objects: 100% (11/11), 2.41 KiB | 1.20 MiB/s, done.
Total 11 (delta 8), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (8/8), completed with 8 local objects.
To https://github.com/Dec32th/laidd-2026.git
   9c89b1d..6d239e6  main -> main
